[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/calculus/13_multiple_integrals_coordinate_transforms/exercises.ipynb)

# Module 13 — Multiple Integrals and Coordinate Transformations — Exercises

Forty fully solved problems across four tiers. Every numeric or algorithmic answer is
recomputed in the code cell that follows its solution; the theorem numbers cited are those of
[`first_principles.ipynb`](first_principles.ipynb).

---

Run this cell first: it is the preamble every later code cell relies on.

In [1]:
import math

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

import sympy as sp
from scipy import integrate, special

print("numpy", np.__version__, "| scipy check:", integrate.quad(lambda t: t, 0, 1)[0])

numpy 2.4.6 | scipy check: 0.5


## L0 — Concept Checks

### Problem L0.1 — Swapping Integration Limits in Rectangular Domain

**Source**: Marsden & Tromba, *Vector Calculus*, Ch. 5.1

**Statement**
Let $R = [0, 2] \times [1, 3] \subset \mathbb{R}^2$ be a rectangular region. Show that for any continuous function $f(x,y)$,

$$
\int_0^2 \int_1^3 f(x,y) \, dy \, dx = \int_1^3 \int_0^2 f(x,y) \, dx \, dy
$$

State the theorem that guarantees this equality.

**Intuition**

For a rectangular domain $[a,b] \times [c,d]$, slicing the 3D volume into slices parallel to the $y$-axis and summing them along the $x$-axis accumulates the exact same total volume as slicing parallel to the $x$-axis and summing along the $y$-axis.

**Solution**

1. The domain $R = \{(x,y) \in \mathbb{R}^2 : 0 \le x \le 2, 1 \le y \le 3\}$ is compact and rectangular.
2. Since $f(x,y)$ is continuous on the compact set $R$, $f$ is bounded and Riemann integrable on $R$.
3. By **Fubini's Theorem for Rectangular Domains** (Theorem 4.1 of `first_principles.ipynb`), the double integral $\iint_R f(x,y) \, dA$ equals both iterated integrals:

$$
\begin{aligned}
\iint_R f(x,y) \, dA &= \int_0^2 \left( \int_1^3 f(x,y) \, dy \right) dx \\
&= \int_1^3 \left( \int_0^2 f(x,y) \, dx \right) dy
\end{aligned}
$$

Hence the two iterated integrals are identically equal.

$$
\boxed{\int_0^2 \int_1^3 f(x,y) \, dy \, dx = \int_1^3 \int_0^2 f(x,y) \, dx \, dy}
$$

**Key takeaway**

For constant integration limits (rectangular domains), swapping the order of integration requires no modification to the limits of integration.

---

In [2]:
# Fubini on a rectangle: both iterated orders of a test f on [0,2] x [1,3].
f = lambda x, y: np.sin(x) * np.exp(-y) + x * y**2
order_yx = integrate.quad(lambda x: integrate.quad(lambda y: f(x, y), 1, 3)[0], 0, 2)[0]
order_xy = integrate.quad(lambda y: integrate.quad(lambda x: f(x, y), 0, 2)[0], 1, 3)[0]
print(f"dy dx = {order_yx:.12f}   dx dy = {order_xy:.12f}   gap = {abs(order_yx - order_xy):.3e}")
assert abs(order_yx - order_xy) < 1e-10

dy dx = 17.783798840809   dx dy = 17.783798840809   gap = 3.553e-15


### Problem L0.2 — Geometric Area Representation via Double Integral

**Source**: Stewart, *Multivariable Calculus*, Ch. 15.2

**Statement**
Express the area $A(D)$ of a bounded planar region $D \subset \mathbb{R}^2$ as a double integral. Explain why setting the integrand $f(x,y) = 1$ yields the area.

**Intuition**

The double integral $\iint_D f(x,y) \, dA$ represents the volume of a 3D solid bounded below by $D$ and above by the surface $z = f(x,y)$. When the height is uniformly $z = 1$, the volume of the cylinder (Height $\times$ Base Area) numerically equals $1 \times \mathrm{Area}(D) = \mathrm{Area}(D)$.

**Solution**

1. Let $P$ be a partition of a rectangle containing $D$ into sub-rectangles $R_k$ of area $\Delta A_k$.
2. The Riemann sum for $f(x,y) = 1$ over $D$ is:

$$
S(1, P) = \sum_{k: R_k \cap D \ne \emptyset} 1 \cdot \Delta A_k = \sum_k \Delta A_k
$$

3. Taking the limit as the mesh size goes to zero:

$$
\lim_{\mathrm{mesh}(P) \to 0} \sum_k \Delta A_k = \iint_D 1 \, dA = \iint_D dx \, dy
$$

$$
\boxed{A(D) = \iint_D dx \, dy}
$$

**Key takeaway**

Integrating the constant function $f=1$ over any $n$-dimensional domain $\Omega \subset \mathbb{R}^n$ computes the $n$-dimensional measure (length for 1D, area for 2D, volume for 3D, hyper-volume for $n$D) of $\Omega$.

---

In [3]:
# Integrating the constant 1 returns the area: unit disk, exact pi.
area, err = integrate.dblquad(lambda y, x: 1.0, -1, 1,
                              lambda x: -np.sqrt(1 - x**2), lambda x: np.sqrt(1 - x**2))
print(f"double integral of 1 over the unit disk = {area:.10f}   pi = {np.pi:.10f}")
assert abs(area - np.pi) < 1e-8

double integral of 1 over the unit disk = 3.1415926536   pi = 3.1415926536


### Problem L0.3 — Geometric Origin of the Polar Area Element $r \, dr \, d\theta$

**Source**: Apostol, *Calculus Vol. II*, Ch. 11.14

**Statement**
In polar coordinates $(r, \theta)$, explain geometrically why the differential area element is $dA = r \, dr \, d\theta$ and not simply $dr \, d\theta$.

**Intuition**

In polar coordinates, grid lines are concentric circles $r = C$ and radial rays $\theta = C$. An infinitesimal polar sector bounded by $[r, r+dr]$ and $[\theta, \theta+d\theta]$ is approximately an infinitesimal rectangle with radial length $dr$ and arc length $r \, d\theta$.

**Solution**

1. The exact area of a circular sector of radius $R$ and angle $\Delta \theta$ is $\frac{1}{2} R^2 \Delta \theta$.
2. The area $\Delta A$ of a polar cell between radii $r$ and $r + \Delta r$ and angles $\theta$ and $\theta + \Delta \theta$ is:

$$
\begin{aligned}
\Delta A &= \frac{1}{2} (r + \Delta r)^2 \Delta \theta - \frac{1}{2} r^2 \Delta \theta \\
&= \frac{1}{2} \left( r^2 + 2r \Delta r + (\Delta r)^2 - r^2 \right) \Delta \theta \\
&= r \, \Delta r \, \Delta \theta + \frac{1}{2} (\Delta r)^2 \Delta \theta
\end{aligned}
$$

3. Taking first-order differentials and dropping higher-order term $(\Delta r)^2 \Delta \theta$:

$$
dA = r \, dr \, d\theta
$$

4. Alternatively, using the Jacobian of $x = r\cos\theta, y = r\sin\theta$:

$$
\det J = \begin{vmatrix} \frac{\partial x}{\partial r} & \frac{\partial x}{\partial \theta} \\ \frac{\partial y}{\partial r} & \frac{\partial y}{\partial \theta} \end{vmatrix} = \begin{vmatrix} \cos\theta & -r\sin\theta \\ \sin\theta & r\cos\theta \end{vmatrix} = r(\cos^2\theta + \sin^2\theta) = r
$$

$$
\boxed{dA = r \, dr \, d\theta}
$$

**Key takeaway**

The extra factor $r$ accounts for the fact that arc length increases linearly with radial distance $r$ from the origin.

---

In [4]:
# The polar Jacobian determinant, symbolically.
r, th = sp.symbols("r theta", positive=True)
T = sp.Matrix([r * sp.cos(th), r * sp.sin(th)])
detJ = sp.simplify(T.jacobian(sp.Matrix([r, th])).det())
print("det J_polar =", detJ)
assert sp.simplify(detJ - r) == 0

det J_polar = r


### Problem L0.4 — Fubini's Theorem Hypothesis Verification

**Source**: Spivak, *Calculus on Manifolds*, Ch. 3

**Statement**
Consider $f(x,y) = \frac{1}{(x-y)^2}$ on $[0,1] \times [0,1]$. Can Fubini's theorem be directly applied to swap the integration order? Explain why or why not.

**Intuition**

Fubini's theorem requires the function to be Riemann integrable (or absolutely integrable under Lebesgue integration). If the function has severe non-integrable singularities inside the domain, Fubini's theorem cannot be applied.

**Solution**

1. Examine $f(x,y) = (x-y)^{-2}$ along the line $y = x$.
2. As $(x,y) \to (x_0, x_0)$, $f(x,y) \to +\infty$.
3. Consider the 1D integral $\int_0^1 \frac{dy}{(x-y)^2} = \left[ \frac{1}{x-y} \right]_0^1 = \frac{1}{x-1} - \left(-\frac{1}{x}\right)$, which diverges at $y=x$ for every $x \in (0,1)$.
4. Since $f(x,y)$ is unbounded and its integral diverges, $f$ is not integrable over $[0,1]^2$.

$$
\boxed{\text{Fubini's theorem cannot be applied because } f \text{ is not integrable on } [0,1]^2.}
$$

**Key takeaway**

Always verify integrability and boundedness (or absolute integrability $\iint \lvert f \rvert dA \lt \infty$) before swapping integration order.

---

In [5]:
# The inner integral of 1/(x-y)^2 over y diverges for every x in (0,1).
x0 = 0.5
mass = lambda eps: (1 / eps - 1 / x0) + (1 / eps - 1 / (1 - x0))
for eps in (1e-1, 1e-2, 1e-3, 1e-4):
    print(f"gap eps = {eps:.0e}   mass of 1/(x-y)^2 outside the gap = {mass(eps):.4e}")
chk = (integrate.quad(lambda y: 1.0 / (x0 - y) ** 2, 0, x0 - 1e-1)[0]
       + integrate.quad(lambda y: 1.0 / (x0 - y) ** 2, x0 + 1e-1, 1)[0])
print(f"quad at eps = 1e-1 gives {chk:.10f}, the formula gives {mass(1e-1):.10f}")
print("the mass grows like 2/eps, so the inner integral is +infinity for every x")
assert abs(chk - mass(1e-1)) < 1e-8

gap eps = 1e-01   mass of 1/(x-y)^2 outside the gap = 1.6000e+01
gap eps = 1e-02   mass of 1/(x-y)^2 outside the gap = 1.9600e+02
gap eps = 1e-03   mass of 1/(x-y)^2 outside the gap = 1.9960e+03
gap eps = 1e-04   mass of 1/(x-y)^2 outside the gap = 1.9996e+04
quad at eps = 1e-1 gives 16.0000000000, the formula gives 16.0000000000
the mass grows like 2/eps, so the inner integral is +infinity for every x


### Problem L0.5 — Coordinate Choice for Quadric Surfaces

**Source**: Marsden & Tromba, *Vector Calculus*, Ch. 6.3

**Statement**
Determine whether Cylindrical or Spherical coordinates are more appropriate for evaluating $\iiint_V z \, dV$, where $V$ is bounded by the cone $z = \sqrt{x^2+y^2}$ and the sphere $x^2+y^2+z^2 = 4$.

**Intuition**

Look at the symmetry of the boundaries:
- Cone $z = \sqrt{x^2+y^2} \implies \rho \cos\phi = \rho \sin\phi \implies \tan\phi = 1 \implies \phi = \frac{\pi}{4}$.
- Sphere $x^2+y^2+z^2 = 4 \implies \rho^2 = 4 \implies \rho = 2$.
Both boundary surfaces become constant coordinate surfaces in Spherical coordinates!

**Solution**

1. In spherical coordinates $(\rho, \theta, \phi)$:
   - Sphere boundary: $\rho = 2$.
   - Cone boundary: $\phi = \frac{\pi}{4}$.
   - Azimuthal angle: $\theta \in [0, 2\pi]$.
2. The domain in spherical coordinates is a rectangular box $[0, 2] \times [0, \frac{\pi}{4}] \times [0, 2\pi]$!
3. In cylindrical coordinates, the upper boundary is $z = \sqrt{4 - r^2}$ and lower boundary is $z = r$, creating a coupled radical limit.

$$
\boxed{\text{Spherical coordinates are optimal because the domain boundaries reduce to constants: } \rho \in [0,2], \, \phi \in [0, \frac{\pi}{4}], \, \theta \in [0, 2\pi].}
$$

**Key takeaway**

When boundary surfaces are cones ($\phi = \text{const}$) and spheres ($\rho = \text{const}$), spherical coordinates transform complex integration limits into constant limits.

---

### Problem L0.6 — Volume of Unit Sphere in Cartesian vs Spherical Coordinates

**Source**: Stewart, *Multivariable Calculus*, Ch. 15.8

**Statement**
Set up the volume integral for the unit sphere $x^2+y^2+z^2 \le 1$ in both Cartesian and Spherical coordinates. Evaluate using Spherical coordinates.

**Intuition**

In Cartesian coordinates, the limits involve nested square roots. In spherical coordinates, all limits are constant.

**Solution**

1. **Cartesian Setup**:

$$
V = \int_{-1}^1 \int_{-\sqrt{1-x^2}}^{\sqrt{1-x^2}} \int_{-\sqrt{1-x^2-y^2}}^{\sqrt{1-x^2-y^2}} 1 \, dz \, dy \, dx
$$

2. **Spherical Setup**:

$$
V = \int_0^{2\pi} \int_0^\pi \int_0^1 \rho^2 \sin\phi \, d\rho \, d\phi \, d\theta
$$

3. **Evaluation**:

$$
\begin{aligned}
V &= \left( \int_0^{2\pi} d\theta \right) \left( \int_0^\pi \sin\phi \, d\phi \right) \left( \int_0^1 \rho^2 d\rho \right) \\
&= (2\pi) \cdot \Big[-\cos\phi\Big]_0^\pi \cdot \left[ \frac{\rho^3}{3} \right]_0^1 \\
&= (2\pi) \cdot (1 - (-1)) \cdot \left( \frac{1}{3} \right) = (2\pi)(2)\left(\frac{1}{3}\right) = \frac{4}{3}\pi
\end{aligned}
$$

$$
\boxed{V = \frac{4}{3}\pi}
$$

**Key takeaway**

Separable integrands over constant coordinate boxes factorize into a product of independent 1D integrals.

---

In [6]:
# Unit-ball volume in spherical parameters versus 4*pi/3.
vol, _ = integrate.nquad(lambda rho, phi, th: rho**2 * np.sin(phi),
                         [[0, 1], [0, np.pi], [0, 2 * np.pi]])
print(f"spherical nquad = {vol:.12f}   4 pi / 3 = {4 * np.pi / 3:.12f}")
assert abs(vol - 4 * np.pi / 3) < 1e-10

spherical nquad = 4.188790204786   4 pi / 3 = 4.188790204786


### Problem L0.7 — Centroid vs Center of Mass for Variable Density

**Source**: Marsden & Tromba, *Vector Calculus*, Ch. 5.5

**Statement**
Define the **centroid** $(\bar{x}, \bar{y})$ and **center of mass** $(x_{cm}, y_{cm})$ of a planar region $D$. Under what condition do they coincide?

**Intuition**

The centroid is the purely geometric center of a shape (assuming uniform unit density). The center of mass is the mass-weighted average position.

**Solution**

1. For density $\rho(x,y)$:
   - Total mass: $M = \iint_D \rho(x,y) \, dA$.
   - Center of mass: $x_{cm} = \frac{1}{M} \iint_D x \, \rho(x,y) \, dA, \quad y_{cm} = \frac{1}{M} \iint_D y \, \rho(x,y) \, dA$.
2. For centroid:
   - Area: $A = \iint_D 1 \, dA$.
   - Centroid coordinates: $\bar{x} = \frac{1}{A} \iint_D x \, dA, \quad \bar{y} = \frac{1}{A} \iint_D y \, dA$.
3. If density is uniform ($\rho(x,y) = \rho_0 = \text{const}$):

$$
M = \iint_D \rho_0 \, dA = \rho_0 A
$$

$$
x_{cm} = \frac{1}{\rho_0 A} \iint_D x \cdot \rho_0 \, dA = \frac{1}{A} \iint_D x \, dA = \bar{x}
$$

$$
\boxed{\text{Centroid and center of mass coincide if and only if the mass density } \rho(x,y) \text{ is uniform (constant).}}
$$

**Key takeaway**

The centroid is a geometric property of the domain, whereas the center of mass depends on the physical density distribution.

---

### Problem L0.8 — Matrix Jacobian vs Determinant Jacobian

**Source**: Spivak, *Calculus on Manifolds*, Ch. 3

**Statement**
Distinguish between the **Jacobian matrix** $J_{\mathbf{T}}(\mathbf{u})$ and the **Jacobian determinant** $\det J_{\mathbf{T}}(\mathbf{u})$. Which one appears in the integral change of variables formula?

**Intuition**

The Jacobian matrix is a linear transformation representing local derivative slopes. The Jacobian determinant is a single scalar measuring volume expansion or contraction.

**Solution**

1. For $\mathbf{T}: \mathbb{R}^n \to \mathbb{R}^n$, the Jacobian matrix $J_{\mathbf{T}}(\mathbf{u})$ is an $n \times n$ matrix of partial derivatives $\left( \frac{\partial x_i}{\partial u_j} \right)$.
2. The Jacobian determinant is the scalar $\det J_{\mathbf{T}}(\mathbf{u})$.
3. In the change of variables formula:

$$
d^n\mathbf{x} = \left\vert \det J_{\mathbf{T}}(\mathbf{u}) \right\vert d^n\mathbf{u}
$$

The absolute value of the determinant is required because volume elements are non-negative scalar quantities.

$$
\boxed{\text{The absolute value of the Jacobian determinant } \lvert\det J_{\mathbf{T}}(\mathbf{u})\rvert \text{ scales differential volume.}}
$$

**Key takeaway**

Never insert a matrix into an integral integrand; only the scalar volume factor $\lvert\det J\rvert$ belongs in differential elements.

---

## L1 — Foundations

### Problem L1.1 — Reversing the Order of Integration

**Source**: Stewart, *Multivariable Calculus*, Ch. 15.2, Ex. 48

**Statement**
Evaluate the double integral by reversing the order of integration:

$$
I = \int_0^1 \int_y^1 e^{x^2} \, dx \, dy
$$

**Intuition**

The inner integral $\int e^{x^2} dx$ cannot be expressed in terms of elementary functions. However, swapping the order of integration will introduce an $x$ factor from the inner $y$-integration, allowing a simple $u$-substitution!

**Solution**

1. **Identify the domain $D$**:
   The given limits specify $0 \le y \le 1$ and $y \le x \le 1$.
   This is a right triangle in the $xy$-plane bounded by $y = 0$, $x = 1$, and the line $y = x$.
2. **Reverse integration order**:
   Describing $D$ with $x$ as the outer variable:
   $x$ ranges from $0$ to $1$. For a fixed $x$, $y$ ranges from $0$ to $x$.
3. **Rewrite integral**:

$$
I = \int_0^1 \int_0^x e^{x^2} \, dy \, dx
$$

4. **Evaluate inner integral**:

$$
\int_0^x e^{x^2} \, dy = e^{x^2} \Big[ y \Big]_0^x = x e^{x^2}
$$

5. **Evaluate outer integral**:

$$
I = \int_0^1 x e^{x^2} \, dx
$$

Substitute $u = x^2 \implies du = 2x \, dx$:

$$
I = \frac{1}{2} \int_0^1 e^u \, du = \frac{1}{2} \left[ e^u \right]_0^1 = \frac{e - 1}{2}
$$

$$
\boxed{I = \frac{e - 1}{2}}
$$

**Key takeaway**

Reversing the integration order is a fundamental tool for evaluating integrals whose immediate anti-derivative is non-elementary.

---

In [7]:
# Reversed order of integration for exp(x^2) over the triangle 0 <= y <= x <= 1.
given = integrate.quad(lambda y: integrate.quad(lambda x: np.exp(x**2), y, 1)[0], 0, 1)[0]
reversed_ = integrate.quad(lambda x: x * np.exp(x**2), 0, 1)[0]
print(f"original = {given:.12f}   reversed = {reversed_:.12f}   (e-1)/2 = {(np.e - 1) / 2:.12f}")
assert abs(given - (np.e - 1) / 2) < 1e-10 and abs(reversed_ - (np.e - 1) / 2) < 1e-12

original = 0.859140914230   reversed = 0.859140914230   (e-1)/2 = 0.859140914230


### Problem L1.2 — Polar Double Integral Over Disk Sector

**Source**: Demidovich, *Problems in Mathematical Analysis*, No. 3901

**Statement**
Evaluate $\iint_D (x^2 + y^2) \, dx \, dy$, where $D$ is the region in the first quadrant bounded by the circle $x^2 + y^2 = 4$ and the coordinate axes.

**Intuition**

The integrand $x^2 + y^2 = r^2$ and the boundary $x^2 + y^2 = 4 \implies r = 2$ both possess pure radial symmetry. Polar coordinates eliminate all Cartesian terms.

**Solution**

1. **Express region $D$ in polar coordinates**:
   First quadrant: $\theta \in [0, \frac{\pi}{2}]$.
   Radius: $r \in [0, 2]$.
2. **Transform integrand and area element**:
   $x^2 + y^2 = r^2$, $dx\,dy = r\,dr\,d\theta$.
3. **Set up integral**:

$$
\begin{aligned}
\iint_D (x^2 + y^2) \, dx \, dy &= \int_0^{\frac{\pi}{2}} \int_0^2 (r^2) \cdot r \, dr \, d\theta \\
&= \left( \int_0^{\frac{\pi}{2}} d\theta \right) \left( \int_0^2 r^3 \, dr \right) \\
&= \left( \frac{\pi}{2} \right) \left( \left[ \frac{r^4}{4} \right]_0^2 \right) \\
&= \left( \frac{\pi}{2} \right) \left( \frac{16}{4} \right) = 2\pi
\end{aligned}
$$

$$
\boxed{\iint_D (x^2 + y^2) \, dx \, dy = 2\pi}
$$

**Key takeaway**

When integrand contains $x^2+y^2$ over circular domains, polar coordinates turn non-linear integrand terms into simple monomials $r^k$.

---

In [8]:
# Quarter disk of radius 2, integrand x^2 + y^2.
val, _ = integrate.nquad(lambda r, th: r**2 * r, [[0, 2], [0, np.pi / 2]])
print(f"polar nquad = {val:.12f}   2 pi = {2 * np.pi:.12f}")
assert abs(val - 2 * np.pi) < 1e-10

polar nquad = 6.283185307180   2 pi = 6.283185307180


### Problem L1.3 — Triple Integral Over a 3D Tetrahedron

**Source**: Marsden & Tromba, *Vector Calculus*, Ch. 5.4

**Statement**
Compute the volume of the solid tetrahedron $T$ in the first octant bounded by the coordinate planes $x=0, y=0, z=0$ and the plane $x + y + z = 1$.

**Intuition**

The region is bounded by linear equations. Setting up Cartesian limits requires finding the 2D footprint in the $xy$-plane by projecting $z=0$.

**Solution**

1. **Determine limits**:
   - $z$ goes from $0$ to $1 - x - y$.
   - In $xy$-plane ($z=0$), $x + y \le 1 \implies y$ goes from $0$ to $1 - x$.
   - $x$ goes from $0$ to $1$.
2. **Set up volume integral**:

$$
V = \int_0^1 \int_0^{1-x} \int_0^{1-x-y} 1 \, dz \, dy \, dx
$$

3. **Evaluate step-by-step**:

$$
\int_0^{1-x-y} dz = 1 - x - y
$$

$$
\int_0^{1-x} (1 - x - y) \, dy = \left[ (1-x)y - \frac{y^2}{2} \right]_0^{1-x} = (1-x)^2 - \frac{(1-x)^2}{2} = \frac{(1-x)^2}{2}
$$

$$
V = \int_0^{1} \frac{(1-x)^2}{2} \, dx = \frac{1}{2} \left[ -\frac{(1-x)^3}{3} \right]_0^1 = \frac{1}{2} \left( 0 - \left(-\frac{1}{3}\right) \right) = \frac{1}{6}
$$

$$
\boxed{V = \frac{1}{6}}
$$

**Key takeaway**

The volume of a standard simplex in $\mathbb{R}^3$ bounded by coordinate planes and $\sum x_i = 1$ is $\frac{1}{3!} = \frac{1}{6}$.

---

In [9]:
# Volume of the standard 3-simplex.
vol, _ = integrate.tplquad(lambda z, y, x: 1.0, 0, 1,
                           lambda x: 0, lambda x: 1 - x,
                           lambda x, y: 0, lambda x, y: 1 - x - y)
print(f"tplquad = {vol:.12f}   1/6 = {1/6:.12f}")
assert abs(vol - 1 / 6) < 1e-12

tplquad = 0.166666666667   1/6 = 0.166666666667


### Problem L1.4 — Surface Area of a Paraboloid

**Source**: Demidovich, *Problems in Mathematical Analysis*, No. 4012

**Statement**
Find the surface area of the portion of the paraboloid $z = x^2 + y^2$ that lies below the plane $z = 1$.

**Intuition**

The surface area formula for $z = f(x,y)$ is $A = \iint_D \sqrt{1 + f_x^2 + f_y^2} \, dA$. Here $f_x = 2x, f_y = 2y$, so $\sqrt{1 + 4(x^2+y^2)}$, perfectly suited for polar integration!

**Solution**

1. **Partial derivatives**:
   $z = x^2 + y^2 \implies \frac{\partial z}{\partial x} = 2x, \quad \frac{\partial z}{\partial y} = 2y$.
2. **Surface area element**:

$$
dS = \sqrt{1 + (2x)^2 + (2y)^2} \, dx \, dy = \sqrt{1 + 4(x^2 + y^2)} \, dx \, dy
$$

3. **Projection domain $D$**:
   $z \le 1 \implies x^2 + y^2 \le 1$. Unit disk in $xy$-plane ($r \in [0,1], \theta \in [0, 2\pi]$).
4. **Evaluate surface integral in polar**:

$$
\begin{aligned}
A &= \int_0^{2\pi} \int_0^1 \sqrt{1 + 4r^2} \, r \, dr \, d\theta \\
&= 2\pi \int_0^1 (1 + 4r^2)^{\frac{1}{2}} r \, dr
\end{aligned}
$$

Substitute $u = 1 + 4r^2 \implies du = 8r \, dr$:

$$
A = 2\pi \cdot \frac{1}{8} \int_1^5 u^{\frac{1}{2}} du = \frac{\pi}{4} \left[ \frac{2}{3} u^{\frac{3}{2}} \right]_1^5 = \frac{\pi}{6} (5\sqrt{5} - 1)
$$

$$
\boxed{A = \frac{\pi}{6} (5\sqrt{5} - 1)}
$$

**Key takeaway**

Surface area integrals over rotationally symmetric graphs $z=f(r)$ reduce to 1D substitution integrals after polar transformation.

---

In [10]:
# Surface area of z = x^2 + y^2 below z = 1.
area, _ = integrate.nquad(lambda r, th: np.sqrt(1 + 4 * r**2) * r, [[0, 1], [0, 2 * np.pi]])
hand = np.pi / 6 * (5 * np.sqrt(5) - 1)
print(f"nquad = {area:.12f}   (pi/6)(5 sqrt5 - 1) = {hand:.12f}")
assert abs(area - hand) < 1e-10

nquad = 5.330413500269   (pi/6)(5 sqrt5 - 1) = 5.330413500269


### Problem L1.5 — Cylindrical Volume Between Cone and Paraboloid

**Source**: Stewart, *Multivariable Calculus*, Ch. 15.7

**Statement**
Evaluate the volume of the region enclosed between the cone $z = \sqrt{x^2+y^2}$ and the inverted paraboloid $z = 2 - x^2 - y^2$.

**Intuition**

The region is bounded by $z_{lower} = r$ and $z_{upper} = 2 - r^2$. Cylindrical coordinates eliminate all angular dependencies.

**Solution**

1. **Find intersection boundary**:
   $r = 2 - r^2 \implies r^2 + r - 2 = 0 \implies (r+2)(r-1) = 0 \implies r = 1$.
   The projection domain $D$ is the unit disk $r \le 1$.
2. **Cylindrical setup**:
   $z \in [r, 2 - r^2], \quad r \in [0, 1], \quad \theta \in [0, 2\pi]$.
3. **Evaluate Volume**:

$$
\begin{aligned}
V &= \int_0^{2\pi} \int_0^1 \int_r^{2-r^2} r \, dz \, dr \, d\theta \\
&= 2\pi \int_0^1 r(2 - r^2 - r) \, dr \\
&= 2\pi \int_0^1 (2r - r^2 - r^3) \, dr \\
&= 2\pi \left[ r^2 - \frac{r^3}{3} - \frac{r^4}{4} \right]_0^1 \\
&= 2\pi \left( 1 - \frac{1}{3} - \frac{1}{4} \right) = 2\pi \left( \frac{5}{12} \right) = \frac{5\pi}{6}
\end{aligned}
$$

$$
\boxed{V = \frac{5\pi}{6}}
$$

**Key takeaway**

Intersection curves of surfaces of revolution $z = f(r)$ define clear radial bounds in cylindrical coordinates.

---

In [11]:
# Volume between the cone z = r and the paraboloid z = 2 - r^2.
vol, _ = integrate.nquad(lambda r, th: r * ((2 - r**2) - r), [[0, 1], [0, 2 * np.pi]])
print(f"nquad = {vol:.12f}   5 pi / 6 = {5 * np.pi / 6:.12f}")
assert abs(vol - 5 * np.pi / 6) < 1e-10

nquad = 2.617993877991   5 pi / 6 = 2.617993877991


### Problem L1.6 — Spherical Evaluation of Ice-Cream Cone Region

**Source**: Marsden & Tromba, *Vector Calculus*, Ch. 6.3

**Statement**
Find the volume of the solid bounded above by the sphere $x^2+y^2+z^2 = 16$ and below by the cone $z = \sqrt{3(x^2+y^2)}$.

**Intuition**

In spherical coordinates, $x^2+y^2+z^2=16 \implies \rho = 4$.
For the cone $z = \sqrt{3(x^2+y^2)} \implies \rho\cos\phi = \sqrt{3} \rho \sin\phi \implies \tan\phi = \frac{1}{\sqrt{3}} \implies \phi = \frac{\pi}{6}$.

**Solution**

1. **Spherical Limits**:
   $\rho \in [0, 4], \quad \phi \in [0, \frac{\pi}{6}], \quad \theta \in [0, 2\pi]$.
2. **Evaluate Integral**:

$$
\begin{aligned}
V &= \int_0^{2\pi} \int_0^{\frac{\pi}{6}} \int_0^4 \rho^2 \sin\phi \, d\rho \, d\phi \, d\theta \\
&= \left( \int_0^{2\pi} d\theta \right) \left( \int_0^{\frac{\pi}{6}} \sin\phi \, d\phi \right) \left( \int_0^4 \rho^2 d\rho \right) \\
&= (2\pi) \cdot \Big[ -\cos\phi \Big]_0^{\frac{\pi}{6}} \cdot \left[ \frac{\rho^3}{3} \right]_0^4 \\
&= (2\pi) \cdot \left( 1 - \frac{\sqrt{3}}{2} \right) \cdot \left( \frac{64}{3} \right) \\
&= \frac{128\pi}{3} \left( 1 - \frac{\sqrt{3}}{2} \right) = \frac{64\pi(2 - \sqrt{3})}{3}
\end{aligned}
$$

$$
\boxed{V = \frac{64\pi(2 - \sqrt{3})}{3}}
$$

**Key takeaway**

Spherical coordinates decompose "ice-cream cone" geometries into completely independent 1D integrals.

---

In [12]:
# Ice-cream cone: sphere rho = 4 above the cone phi = pi/6.
vol, _ = integrate.nquad(lambda rho, phi, th: rho**2 * np.sin(phi),
                         [[0, 4], [0, np.pi / 6], [0, 2 * np.pi]])
hand = 64 * np.pi * (2 - np.sqrt(3)) / 3
print(f"nquad = {vol:.12f}   64 pi (2 - sqrt3)/3 = {hand:.12f}")
assert abs(vol - hand) < 1e-10

nquad = 17.958127242175   64 pi (2 - sqrt3)/3 = 17.958127242175


### Problem L1.7 — Linear Coordinate Change Over Elliptic Disk

**Source**: Demidovich, *Problems in Mathematical Analysis*, No. 3935

**Statement**
Evaluate $\iint_E (x+y)^2 \, dx \, dy$, where $E$ is the elliptic region $\frac{x^2}{a^2} + \frac{y^2}{b^2} \le 1$.

**Intuition**

Transform the ellipse into a unit circle using scaled coordinates $u = \frac{x}{a}, v = \frac{y}{b}$, then apply polar coordinates in $(u,v)$-space!

**Solution**

1. **Transformation**:
   Let $x = a u, \quad y = b v$.
   The domain becomes $u^2 + v^2 \le 1$ (unit disk $D$).
2. **Jacobian**:

$$
J = \begin{pmatrix} a & 0 \\ 0 & b \end{pmatrix} \implies \det J = ab \implies dx\,dy = ab \, du\,dv
$$

3. **Integrand transformation**:
   $(x+y)^2 = (au + bv)^2 = a^2 u^2 + 2ab uv + b^2 v^2$.
4. **Transform $(u,v)$ to Polar $(r, \theta)$**:
   $u = r\cos\theta, v = r\sin\theta, du\,dv = r\,dr\,d\theta$.

$$
(au + bv)^2 = r^2 (a\cos\theta + b\sin\theta)^2 = r^2 (a^2 \cos^2\theta + 2ab\sin\theta\cos\theta + b^2 \sin^2\theta)
$$

5. **Evaluate Integral**:

$$
\iint_D (a^2 u^2 + 2ab uv + b^2 v^2) ab \, du \, dv = ab \int_0^{2\pi} \int_0^1 r^3 (a^2 \cos^2\theta + 2ab\sin\theta\cos\theta + b^2 \sin^2\theta) \, dr \, d\theta
$$

Note that $\int_0^{2\pi} \cos^2\theta \, d\theta = \pi$, $\int_0^{2\pi} \sin^2\theta \, d\theta = \pi$, and $\int_0^{2\pi} \sin\theta\cos\theta \, d\theta = 0$.

$$
\begin{aligned}
I &= ab \left( \int_0^1 r^3 dr \right) \left( a^2 \pi + 0 + b^2 \pi \right) \\
&= ab \left( \frac{1}{4} \right) \pi (a^2 + b^2) = \frac{\pi ab (a^2 + b^2)}{4}
\end{aligned}
$$

$$
\boxed{\iint_E (x+y)^2 \, dx \, dy = \frac{\pi ab (a^2 + b^2)}{4}}
$$

**Key takeaway**

Combining linear diagonal scaling with polar transformation easily maps elliptic domains to unit disks.

---

In [13]:
# (x+y)^2 over an ellipse with a = 3, b = 2.
a, b = 3.0, 2.0
val, _ = integrate.nquad(lambda r, th: (a * r * np.cos(th) + b * r * np.sin(th)) ** 2 * a * b * r,
                         [[0, 1], [0, 2 * np.pi]])
hand = np.pi * a * b * (a**2 + b**2) / 4
print(f"nquad = {val:.10f}   pi a b (a^2 + b^2)/4 = {hand:.10f}")
assert abs(val - hand) < 1e-8

nquad = 61.2610567450   pi a b (a^2 + b^2)/4 = 61.2610567450


### Problem L1.8 — Improper Double Integral Over $\mathbb{R}^2$

**Source**: Demidovich, *Problems in Mathematical Analysis*, No. 3965

**Statement**
Evaluate the improper double integral over the entire 2D plane:

$$
I = \iint_{\mathbb{R}^2} \frac{dx \, dy}{(1 + x^2 + y^2)^2}
$$

**Intuition**

The domain $\mathbb{R}^2$ and integrand are radially symmetric. Convert to polar coordinates with limits $r \in [0, \infty)$ and $\theta \in [0, 2\pi]$.

**Solution**

1. **Polar Transformation**:

$$
I = \int_0^{2\pi} \int_0^\infty \frac{r \, dr \, d\theta}{(1 + r^2)^2} = 2\pi \int_0^\infty \frac{r}{(1 + r^2)^2} \, dr
$$

2. **Substitution $u = 1 + r^2 \implies du = 2r \, dr$**:

$$
I = 2\pi \cdot \frac{1}{2} \int_1^\infty \frac{du}{u^2} = \pi \left[ -\frac{1}{u} \right]_1^\infty = \pi (0 - (-1)) = \pi
$$

$$
\boxed{I = \pi}
$$

**Key takeaway**

Improper multivariable integrals over unbounded domains converge smoothly when transformed to polar coordinates if radial decay is sufficiently fast ($\gt \frac{1}{r^2}$).

---

In [14]:
# Improper integral of (1 + x^2 + y^2)^{-2} over the plane.
val, _ = integrate.nquad(lambda r, th: r / (1 + r**2) ** 2, [[0, np.inf], [0, 2 * np.pi]])
print(f"nquad = {val:.12f}   pi = {np.pi:.12f}")
assert abs(val - np.pi) < 1e-9

nquad = 3.141592653590   pi = 3.141592653590


### Problem L1.9 — Center of Mass of a Solid Hemisphere

**Source**: Marsden & Tromba, *Vector Calculus*, Ch. 5.5

**Statement**
Find the center of mass of a solid hemisphere $H = \{(x,y,z) \in \mathbb{R}^3 : x^2+y^2+z^2 \le R^2, z \ge 0\}$ of uniform density $\rho_0$.

**Intuition**

By rotational symmetry about the $z$-axis, $\bar{x} = 0$ and $\bar{y} = 0$. We only need to calculate $\bar{z}$.

**Solution**

1. **Total mass $M$**:
   Volume of hemisphere is $\frac{2}{3}\pi R^3$, so $M = \frac{2}{3}\rho_0 \pi R^3$.
2. **Calculate mass moment $M_{xy} = \iiint_H z \rho_0 \, dV$**:
   In spherical coordinates: $z = \rho \cos\phi, dV = \rho^2 \sin\phi \, d\rho \, d\phi \, d\theta$.
   Limits: $\rho \in [0, R], \phi \in [0, \frac{\pi}{2}], \theta \in [0, 2\pi]$.

$$
\begin{aligned}
M_{xy} &= \rho_0 \int_0^{2\pi} \int_0^{\frac{\pi}{2}} \int_0^R (\rho \cos\phi) \rho^2 \sin\phi \, d\rho \, d\phi \, d\theta \\
&= \rho_0 (2\pi) \left( \int_0^{\frac{\pi}{2}} \sin\phi \cos\phi \, d\phi \right) \left( \int_0^R \rho^3 \, d\rho \right) \\
&= \rho_0 (2\pi) \left[ \frac{\sin^2\phi}{2} \right]_0^{\frac{\pi}{2}} \left[ \frac{R^4}{4} \right] \\
&= \rho_0 (2\pi) \left( \frac{1}{2} \right) \left( \frac{R^4}{4} \right) = \frac{\rho_0 \pi R^4}{4}
\end{aligned}
$$

3. **Calculate $\bar{z}$**:

$$
\bar{z} = \frac{M_{xy}}{M} = \frac{\frac{\rho_0 \pi R^4}{4}}{\frac{2}{3}\rho_0 \pi R^3} = \frac{3}{8} R
$$

$$
\boxed{(\bar{x}, \bar{y}, \bar{z}) = \left(0, 0, \frac{3}{8} R\right)}
$$

**Key takeaway**

Symmetry eliminates components of the center of mass, reducing 3D calculations to a single directional moment.

---

In [15]:
# Centre of mass of a uniform solid hemisphere of radius R = 2.
R = 2.0
mass, _ = integrate.nquad(lambda rho, phi, th: rho**2 * np.sin(phi),
                          [[0, R], [0, np.pi / 2], [0, 2 * np.pi]])
mom, _ = integrate.nquad(lambda rho, phi, th: rho * np.cos(phi) * rho**2 * np.sin(phi),
                         [[0, R], [0, np.pi / 2], [0, 2 * np.pi]])
print(f"z-bar = {mom / mass:.12f}   3R/8 = {3 * R / 8:.12f}")
assert abs(mom / mass - 3 * R / 8) < 1e-10

z-bar = 0.750000000000   3R/8 = 0.750000000000


### Problem L1.10 — Moment of Inertia of a Solid Cylinder

**Source**: Marsden & Tromba, *Vector Calculus*, Ch. 5.5

**Statement**
Find the moment of inertia $I_z$ of a solid right circular cylinder of radius $R$, height $h$, and uniform mass $M$ about its central longitudinal axis ($z$-axis).

**Intuition**

The moment of inertia about the $z$-axis measures resistance to rotational acceleration and is defined as $I_z = \iiint r^2 \rho_0 \, dV$, where $r^2 = x^2 + y^2$.

**Solution**

1. **Density**:
   $\rho_0 = \frac{M}{V} = \frac{M}{\pi R^2 h}$.
2. **Cylindrical Setup**:

$$
I_z = \iiint_V (x^2+y^2) \rho_0 \, dV = \rho_0 \int_0^h \int_0^{2\pi} \int_0^R (r^2) \cdot r \, dr \, d\theta \, dz
$$

3. **Evaluate**:

$$
\begin{aligned}
I_z &= \rho_0 h (2\pi) \int_0^R r^3 \, dr \\
&= \rho_0 h (2\pi) \left( \frac{R^4}{4} \right) = \frac{\pi \rho_0 h R^4}{2}
\end{aligned}
$$

4. **Substitute $\rho_0 = \frac{M}{\pi R^2 h}$**:

$$
I_z = \frac{\pi h R^4}{2} \left( \frac{M}{\pi R^2 h} \right) = \frac{1}{2} M R^2
$$

$$
\boxed{I_z = \frac{1}{2} M R^2}
$$

**Key takeaway**

The extra factor of $r^2$ in the moment of inertia integrand means mass further from the axis contributes quadratically more to rotational inertia.

---

In [16]:
# Moment of inertia of a uniform solid cylinder, R = 1.5, h = 4, M = 7.
R, h, M = 1.5, 4.0, 7.0
dens = M / (np.pi * R**2 * h)
Iz, _ = integrate.nquad(lambda r, th, z: dens * r**2 * r, [[0, R], [0, 2 * np.pi], [0, h]])
print(f"I_z = {Iz:.12f}   M R^2 / 2 = {0.5 * M * R**2:.12f}")
assert abs(Iz - 0.5 * M * R**2) < 1e-10

I_z = 7.875000000000   M R^2 / 2 = 7.875000000000


### Problem L1.11 — Jacobian of Hyperbolic / Conformal Transformation

**Source**: Apostol, *Calculus Vol. II*, Ch. 12.5

**Statement**
Compute the Jacobian determinant of the transformation $u = x^2 - y^2, v = 2xy$. Determine where the transformation is locally invertible.

**Intuition**

This transformation corresponds to the complex mapping $w = z^2$ (where $z = x+iy$). By Cauchy-Riemann equations, its local area distortion equals $\lvert f'(z) \rvert^2 = \lvert 2z \rvert^2 = 4(x^2+y^2)$.

**Solution**

1. **Compute Jacobian Matrix $J_{(x,y)}$**:

$$
J = \begin{pmatrix} \frac{\partial u}{\partial x} & \frac{\partial u}{\partial y} \\ \frac{\partial v}{\partial x} & \frac{\partial v}{\partial y} \end{pmatrix} = \begin{pmatrix} 2x & -2y \\ 2y & 2x \end{pmatrix}
$$

2. **Compute Determinant**:

$$
\det J = (2x)(2x) - (-2y)(2y) = 4x^2 + 4y^2 = 4(x^2 + y^2)
$$

3. **Invertibility Condition**:
   By the Inverse Function Theorem, the transformation is locally $C^1$-invertible at any point where $\det J \ne 0$.
   $\det J = 0 \iff x^2 + y^2 = 0 \iff (x,y) = (0,0)$.

$$
\boxed{\det J = 4(x^2+y^2). \text{ The transformation is locally invertible everywhere except at the origin } (0,0).}
$$

**Key takeaway**

Conformal maps preserve local angles and scale differential areas by $\lvert f'(z) \rvert^2$.

---

In [17]:
# Jacobian determinant of u = x^2 - y^2, v = 2xy.
x, y = sp.symbols("x y", real=True)
F = sp.Matrix([x**2 - y**2, 2 * x * y])
detJ = sp.simplify(F.jacobian(sp.Matrix([x, y])).det())
print("det J =", sp.factor(detJ), "   zero set:", sp.solve([detJ], [x, y], dict=True))
assert sp.simplify(detJ - 4 * (x**2 + y**2)) == 0

det J = 4*(x**2 + y**2)    zero set: [{x: -I*y}, {x: I*y}]


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Newton's Shell Theorem for Gravitational Potential

**Source**: Classical Mechanics & Marsden & Tromba, *Vector Calculus*, Ch. 6.3

**Statement**
A thin spherical shell of radius $R$ has uniform mass density $\sigma_0$. Prove that the gravitational potential $\Phi(r)$ at a point $P$ at distance $r \gt R$ from the center is identical to that of a point mass $M = 4\pi R^2 \sigma_0$ located at the origin.

**Intuition**

Summing the individual gravitational attractions of every microscopic patch on the spherical surface via 2D spherical integration causes non-radial force components to cancel completely, leaving an effective single point mass potential.

**Solution**

1. Place point $P$ on the positive $z$-axis at distance $r$ from the origin: $\mathbf{r}_P = (0,0,r)$.
2. Point on shell of radius $R$: $\mathbf{r}' = (R\sin\phi\cos\theta, R\sin\phi\sin\theta, R\cos\phi)$.
3. Distance $d$ between $P$ and $\mathbf{r}'$:

$$
d = \|\mathbf{r}_P - \mathbf{r}'\| = \sqrt{R^2\sin^2\phi + (r - R\cos\phi)^2} = \sqrt{r^2 + R^2 - 2rR\cos\phi}
$$

4. Surface element $dS = R^2 \sin\phi \, d\phi \, d\theta$. Total mass $M = 4\pi R^2 \sigma_0$.
5. Gravitational potential integral:

$$
\Phi(r) = -G \sigma_0 \iint_S \frac{dS}{d} = -G \sigma_0 R^2 \int_0^{2\pi} d\theta \int_0^\pi \frac{\sin\phi}{\sqrt{r^2 + R^2 - 2rR\cos\phi}} d\phi
$$

6. Evaluate inner integral via $u = r^2 + R^2 - 2rR\cos\phi \implies du = 2rR\sin\phi \, d\phi$:
   - For $\phi = 0 \implies u = (r-R)^2 \implies \sqrt{u} = r-R$ (since $r \gt R$).
   - For $\phi = \pi \implies u = (r+R)^2 \implies \sqrt{u} = r+R$.

$$
\int_0^\pi \frac{\sin\phi}{\sqrt{r^2 + R^2 - 2rR\cos\phi}} d\phi = \frac{1}{2rR} \int_{(r-R)^2}^{(r+R)^2} u^{-\frac{1}{2}} du = \frac{1}{2rR} \Big[ 2\sqrt{u} \Big]_{(r-R)^2}^{(r+R)^2} = \frac{1}{rR} ((r+R) - (r-R)) = \frac{2}{r}
$$

7. Substitute back:

$$
\Phi(r) = -G \sigma_0 R^2 (2\pi) \left( \frac{2}{r} \right) = -\frac{G (4\pi R^2 \sigma_0)}{r} = -\frac{G M}{r}
$$

$$
\boxed{\Phi(r) = -\frac{GM}{r}}
$$

**Key takeaway**

Spherical surface integrals mathematically justify treating spherically symmetric planets as point masses in orbital mechanics.

---

In [18]:
# Shell theorem: numerical surface integral versus -GM/r, with G = sigma0 = 1.
R, r_out = 1.0, 3.0
inner, _ = integrate.nquad(
    lambda phi, th: np.sin(phi) / np.sqrt(r_out**2 + R**2 - 2 * r_out * R * np.cos(phi)),
    [[0, np.pi], [0, 2 * np.pi]])
Phi = -R**2 * inner
M = 4 * np.pi * R**2
print(f"numeric Phi(r) = {Phi:.12f}   -G M / r = {-M / r_out:.12f}")
assert abs(Phi + M / r_out) < 1e-10

numeric Phi(r) = -4.188790204786   -G M / r = -4.188790204786


### Problem L2.2 — Normalizing Flow Probability Density Transformation

**Source**: Machine Learning Theory / RealNVP (Dinh et al., 2017)

**Statement**
In a RealNVP Affine Coupling Layer, input $\mathbf{z} = (z_1, z_2)^T \in \mathbb{R}^2$ is mapped to output $\mathbf{x} = (x_1, x_2)^T$ via:

$$
x_1 = z_1, \quad x_2 = z_2 \exp(s(z_1)) + t(z_1)
$$

where $s$ and $t$ are arbitrary neural network functions.
1. Derive the Jacobian matrix $J_f(\mathbf{z})$ and its determinant.
2. Given base distribution $p_Z(\mathbf{z}) = \mathcal{N}(\mathbf{0}, \mathbf{I})$, write the exact expression for log-likelihood $\log p_X(\mathbf{x})$.

**Intuition**

Normalizing flows transform simple distributions into complex data distributions. By designing triangular Jacobians, evaluating the determinant reduces to summing diagonal scale terms $\exp(s(z_1))$, making high-dimensional likelihood calculation $O(N)$.

**Solution**

1. **Compute Jacobian Matrix $J_f(\mathbf{z})$**:

$$
J_f(\mathbf{z}) = \begin{pmatrix} \frac{\partial x_1}{\partial z_1} & \frac{\partial x_1}{\partial z_2} \\ \frac{\partial x_2}{\partial z_1} & \frac{\partial x_2}{\partial z_2} \end{pmatrix} = \begin{pmatrix} 1 & 0 \\ z_2 s'(z_1)\exp(s(z_1)) + t'(z_1) & \exp(s(z_1)) \end{pmatrix}
$$

2. **Compute Determinant**:
   Since $J_f(\mathbf{z})$ is lower triangular:

$$
\det J_f(\mathbf{z}) = 1 \cdot \exp(s(z_1)) = \exp(s(z_1))
$$

3. **Change of Variables Log-Likelihood**:
   Inverse mapping: $z_1 = x_1, \quad z_2 = (x_2 - t(x_1))\exp(-s(x_1))$.

$$
p_X(\mathbf{x}) = p_Z(\mathbf{z}(\mathbf{x})) \cdot \left\vert \det J_f(\mathbf{z}(\mathbf{x})) \right\vert^{-1}
$$

$$
\log p_X(\mathbf{x}) = \log p_Z(\mathbf{z}(\mathbf{x})) - s(x_1)
$$

For standard Gaussian $p_Z(\mathbf{z}) = \frac{1}{2\pi} \exp\left(-\frac{z_1^2 + z_2^2}{2}\right)$:

$$
\boxed{\log p_X(\mathbf{x}) = -\log(2\pi) - \frac{x_1^2 + (x_2 - t(x_1))^2 e^{-2s(x_1)}}{2} - s(x_1)}
$$

**Key takeaway**

Triangular Jacobian structures allow deep neural networks to compute exact high-dimensional probability density integrals efficiently.

---

In [19]:
# RealNVP coupling layer: hand-rolled log|det J| versus a finite-difference Jacobian.
s_fun, t_fun = lambda z1: 0.7 * np.tanh(z1) + 0.2, lambda z1: 0.5 * z1**2
fwd = lambda z: np.array([z[0], z[1] * np.exp(s_fun(z[0])) + t_fun(z[0])])
z = np.array([0.4, -1.1])
eps = 1e-6
Jnum = np.column_stack([(fwd(z + eps * e) - fwd(z - eps * e)) / (2 * eps)
                        for e in np.eye(2)])
sign, logdet = np.linalg.slogdet(Jnum)
print("finite-difference J =\n", Jnum)
print(f"log|det J| numeric = {logdet:.10f}   s(z1) = {s_fun(z[0]):.10f}")
assert abs(logdet - s_fun(z[0])) < 1e-6 and sign > 0

# The boxed log-density, checked against the pushforward of a standard normal by sampling.
x = fwd(z)
logp = -np.log(2 * np.pi) - 0.5 * (x[0] ** 2 + (x[1] - t_fun(x[0])) ** 2 * np.exp(-2 * s_fun(x[0]))) - s_fun(x[0])
direct = (-np.log(2 * np.pi) - 0.5 * (z @ z)) - logdet
print(f"boxed formula = {logp:.10f}   log p_Z(z) - log|det J| = {direct:.10f}")
assert abs(logp - direct) < 1e-9

finite-difference J =
 [[ 1.      0.    ]
 [-0.6499  1.5936]]
log|det J| numeric = 0.4659642735   s(z1) = 0.4659642736
boxed formula = -2.9888413400   log p_Z(z) - log|det J| = -2.9888413399


### Problem L2.3 — Multivariate Gaussian Integral with Linear Shift

**Source**: Machine Learning & Quantum Field Theory / Bishop, *PRML*, Ch. 2

**Statement**
Evaluate the $n$-dimensional Gaussian integral with linear term $\mathbf{b} \in \mathbb{R}^n$:

$$
I = \int_{\mathbb{R}^n} \exp\left( -\frac{1}{2} \mathbf{x}^T \mathbf{A} \mathbf{x} + \mathbf{b}^T \mathbf{x} \right) d^n\mathbf{x}
$$

where $\mathbf{A} \in \mathbb{R}^{n \times n}$ is a symmetric positive-definite matrix.

**Intuition**

Complete the square in the exponent to convert the integrand into a standard centered Gaussian integral multiplied by a constant factor $e^{\frac{1}{2}\mathbf{b}^T \mathbf{A}^{-1}\mathbf{b}}$.

**Solution**

1. **Complete the square**:
   Let $\mathbf{x} = \mathbf{y} + \mathbf{x}_0$.

$$
\begin{aligned}
-\frac{1}{2} \mathbf{x}^T \mathbf{A} \mathbf{x} + \mathbf{b}^T \mathbf{x} &= -\frac{1}{2} (\mathbf{y} + \mathbf{x}_0)^T \mathbf{A} (\mathbf{y} + \mathbf{x}_0) + \mathbf{b}^T (\mathbf{y} + \mathbf{x}_0) \\
&= -\frac{1}{2} \mathbf{y}^T \mathbf{A} \mathbf{y} - \mathbf{y}^T \mathbf{A} \mathbf{x}_0 + \mathbf{b}^T \mathbf{y} - \frac{1}{2} \mathbf{x}_0^T \mathbf{A} \mathbf{x}_0 + \mathbf{b}^T \mathbf{x}_0
\end{aligned}
$$
2. Set the linear term in $\mathbf{y}$ to zero: $-\mathbf{A} \mathbf{x}_0 + \mathbf{b} = \mathbf{0} \implies \mathbf{x}_0 = \mathbf{A}^{-1} \mathbf{b}$.
3. Substitute $\mathbf{x}_0$:

$$
-\frac{1}{2} \mathbf{x}_0^T \mathbf{A} \mathbf{x}_0 + \mathbf{b}^T \mathbf{x}_0 = -\frac{1}{2} (\mathbf{A}^{-1}\mathbf{b})^T \mathbf{A} (\mathbf{A}^{-1}\mathbf{b}) + \mathbf{b}^T \mathbf{A}^{-1}\mathbf{b} = \frac{1}{2} \mathbf{b}^T \mathbf{A}^{-1} \mathbf{b}
$$
4. Substitute back into integral (Jacobian of translation $\mathbf{x} \to \mathbf{y}$ is $1$):

$$
I = \exp\left( \frac{1}{2} \mathbf{b}^T \mathbf{A}^{-1} \mathbf{b} \right) \int_{\mathbb{R}^n} \exp\left( -\frac{1}{2} \mathbf{y}^T \mathbf{A} \mathbf{y} \right) d^n\mathbf{y}
$$
5. Using Theorem 4.5 of `first_principles.ipynb` for the standard multivariate Gaussian integral:

$$
\int_{\mathbb{R}^n} \exp\left( -\frac{1}{2} \mathbf{y}^T \mathbf{A} \mathbf{y} \right) d^n\mathbf{y} = \sqrt{\frac{(2\pi)^n}{\det \mathbf{A}}}
$$

$$
\boxed{I = \sqrt{\frac{(2\pi)^n}{\det \mathbf{A}}} \exp\left( \frac{1}{2} \mathbf{b}^T \mathbf{A}^{-1} \mathbf{b} \right)}
$$

**Key takeaway**

Completing the square in multivariable space is equivalent to shifting the integration origin to the mode of the Gaussian distribution.

---

In [20]:
# Gaussian integral with a linear term, n = 2, against 2-D quadrature.
A = np.array([[2.0, 0.5], [0.5, 1.5]])
b = np.array([0.3, -0.8])
num, _ = integrate.dblquad(
    lambda y, x: np.exp(-0.5 * (A[0, 0] * x * x + 2 * A[0, 1] * x * y + A[1, 1] * y * y)
                        + b[0] * x + b[1] * y), -40, 40, -40, 40)
hand = np.sqrt((2 * np.pi) ** 2 / np.linalg.det(A)) * np.exp(0.5 * b @ np.linalg.solve(A, b))
print(f"dblquad = {num:.10f}   closed form = {hand:.10f}")
assert abs(num - hand) < 1e-8

dblquad = 5.1191361365   closed form = 5.1191361365


### Problem L2.4 — Moment of Inertia Tensor of an Ellipsoid

**Source**: Physics / Demidovich No. 4105

**Statement**
Compute the principal moment of inertia $I_z = \iiint_V (x^2+y^2) \rho_0 \, dV$ of a solid ellipsoid $\frac{x^2}{a^2} + \frac{y^2}{b^2} + \frac{z^2}{c^2} \le 1$ of total mass $M$ and uniform density $\rho_0$.

**Intuition**

Apply linear scaling to transform the ellipsoid into a unit sphere, then use spherical coordinates.

**Solution**

1. **Transformation to Spherical Coordinates**:
   $x = a \rho \sin\phi \cos\theta, \quad y = b \rho \sin\phi \sin\theta, \quad z = c \rho \cos\phi$.
   Jacobian determinant: $\det J = a b c \, \rho^2 \sin\phi$.
   Domain: $\rho \in [0,1], \phi \in [0,\pi], \theta \in [0,2\pi]$.
2. **Total Mass $M$**:

$$
\iiint_V \rho_0 dV = \rho_0 a b c \int_0^{2\pi} d\theta \int_0^\pi \sin\phi d\phi \int_0^1 \rho^2 d\rho = \frac{4}{3} \pi a b c \rho_0
$$

3. **Compute $I_z$**:

$$
x^2 + y^2 = \rho^2 \sin^2\phi \left( a^2 \cos^2\theta + b^2 \sin^2\theta \right)
$$

$$
I_z = \rho_0 a b c \int_0^{2\pi} (a^2 \cos^2\theta + b^2 \sin^2\theta) d\theta \int_0^\pi \sin^3\phi d\phi \int_0^1 \rho^4 d\rho
$$

   - $\int_0^{2\pi} (a^2 \cos^2\theta + b^2 \sin^2\theta) d\theta = \pi(a^2 + b^2)$.
   - $\int_0^\pi \sin^3\phi d\phi = \int_0^\pi (1 - \cos^2\phi)\sin\phi d\phi = \left[ -\cos\phi + \frac{\cos^3\phi}{3} \right]_0^\pi = \frac{4}{3}$.
   - $\int_0^1 \rho^4 d\rho = \frac{1}{5}$.

4. Multiply terms:

$$
I_z = \rho_0 a b c \cdot \pi(a^2 + b^2) \cdot \frac{4}{3} \cdot \frac{1}{5} = \frac{4}{15} \pi \rho_0 a b c (a^2 + b^2)
$$

5. Substitute $M = \frac{4}{3}\pi a b c \rho_0$:

$$
I_z = \frac{1}{5} M (a^2 + b^2)
$$

$$
\boxed{I_z = \frac{1}{5} M (a^2 + b^2)}
$$

**Key takeaway**

Rescaling coordinates converts anisotropic physical bodies into isotropic spheres, simplifying moment of inertia tensors.

---

In [21]:
# Inertia of a uniform solid ellipsoid, a=3, b=2, c=1, M=5.
a, b, c, M = 3.0, 2.0, 1.0, 5.0
dens = M / (4 / 3 * np.pi * a * b * c)
Iz, _ = integrate.nquad(
    lambda rho, phi, th: dens * a * b * c * rho**4 * np.sin(phi) ** 3
    * (a**2 * np.cos(th) ** 2 + b**2 * np.sin(th) ** 2),
    [[0, 1], [0, np.pi], [0, 2 * np.pi]])
print(f"I_z = {Iz:.12f}   M (a^2 + b^2)/5 = {M * (a**2 + b**2) / 5:.12f}")
assert abs(Iz - M * (a**2 + b**2) / 5) < 1e-9

I_z = 13.000000000000   M (a^2 + b^2)/5 = 13.000000000000


### Problem L2.5 — Monte Carlo Estimation of High-Dimensional Volume

**Source**: Computational AI/ML & Statistical Simulation

**Statement**
Consider estimating the volume $V$ of a 10-dimensional hypersphere of radius $R=1$ enclosed inside a hypercube $[-1, 1]^{10}$.
1. What is the theoretical acceptance ratio $P = \frac{V(B_{10})}{V(Cube_{10})}$ of uniform random sampling?
2. Explain why rejection sampling fails in high dimensions.

**Intuition**

As dimension $n$ increases, almost all volume of a hypercube concentrates in its corners outside the embedded hypersphere.

**Solution**

1. **Hypercube Volume**: $V(Cube_{10}) = 2^{10} = 1024$.
2. **Hypersphere Volume**: Using $V_n(1) = \frac{\pi^{\frac{n}{2}}}{\Gamma\left(\frac{n}{2}+1\right)}$ for $n=10$:

$$
V(B_{10}) = \frac{\pi^5}{\Gamma(6)} = \frac{\pi^5}{120} \approx \frac{306.019}{120} \approx 2.55015
$$

3. **Acceptance Ratio $P$**:

$$
P = \frac{V(B_{10})}{V(Cube_{10})} = \frac{2.55015}{1024} \approx 0.00249 \quad (\approx 0.25\%)
$$
4. In $n=20$ dimensions, $P \approx 2.46 \times 10^{-8}$. Out of 100 million samples, only 2 points hit the sphere!

$$
\boxed{P = \frac{\pi^5}{122880} \approx 0.00249. \text{ Rejection sampling fails due to exponential concentration of volume in hypercube corners.}}
$$

**Key takeaway**

Naive Monte Carlo uniform rejection sampling suffers from the curse of dimensionality; specialized importance sampling or Markov Chain Monte Carlo (MCMC) is mandatory in high dimensions.

---

In [22]:
# Acceptance rate of rejection sampling for B_10(1) inside [-1,1]^10.
P10 = np.pi**5 / 120 / 2**10
P20 = np.exp(10 * np.log(np.pi) - special.gammaln(11)) / 2**20
print(f"pi^5 / 122880 = {P10:.6e}     n = 20 rate = {P20:.6e}")
hits = ((rng.uniform(-1, 1, size=(2_000_000, 10)) ** 2).sum(axis=1) <= 1.0).mean()
print(f"empirical rate at n = 10 (2e6 samples) = {hits:.6e}")
assert abs(P10 - np.pi**5 / 122880) < 1e-15 and abs(hits - P10) < 5e-4

pi^5 / 122880 = 2.490395e-03     n = 20 rate = 2.461137e-08


empirical rate at n = 10 (2e6 samples) = 2.440000e-03


### Problem L2.6 — Continuous Normalizing Flows (CNF) & Instantaneous Change of Variables

**Source**: Neural ODEs (Chen et al., NeurIPS 2018)

**Statement**
Let a probability density $p(\mathbf{z}(t), t)$ evolve under a continuous dynamics defined by Ordinary Differential Equation (ODE) $\frac{d\mathbf{z}}{dt} = \mathbf{f}(\mathbf{z}(t), t)$. Prove that the instantaneous change in log-density is given by:

$$
\frac{d \log p(\mathbf{z}(t), t)}{dt} = -\operatorname{Tr}\left( J_{\mathbf{f}}(\mathbf{z}(t)) \right) = -\nabla \cdot \mathbf{f}(\mathbf{z}(t), t)
$$

**Intuition**

Probability mass conservation in continuous space is governed by the Continuity Equation in fluid dynamics: $\frac{\partial p}{\partial t} + \nabla \cdot (p \mathbf{f}) = 0$.

**Solution**

1. **Continuity Equation**:

$$
\frac{\partial p}{\partial t} + \nabla \cdot (p \mathbf{f}) = 0 \implies \frac{\partial p}{\partial t} + \mathbf{f} \cdot \nabla p + p (\nabla \cdot \mathbf{f}) = 0
$$

2. **Material Derivative of $p$**:
   Along trajectory $\mathbf{z}(t)$, the total time derivative is:

$$
\frac{dp(\mathbf{z}(t), t)}{dt} = \frac{\partial p}{\partial t} + \nabla p \cdot \frac{d\mathbf{z}}{dt} = \frac{\partial p}{\partial t} + \mathbf{f} \cdot \nabla p
$$

3. Substitute continuity equation:

$$
\frac{dp}{dt} = -p (\nabla \cdot \mathbf{f})
$$

4. Divide by $p$:

$$
\frac{d \log p}{dt} = \frac{1}{p} \frac{dp}{dt} = -\nabla \cdot \mathbf{f} = -\operatorname{Tr}\left( J_{\mathbf{f}}(\mathbf{z}(t)) \right)
$$

$$
\boxed{\frac{d \log p(\mathbf{z}(t), t)}{dt} = -\operatorname{Tr}\left( J_{\mathbf{f}}(\mathbf{z}(t)) \right)}
$$

**Key takeaway**

Continuous normalizing flows replace expensive $O(n^3)$ determinant calculations with $O(n)$ Jacobian trace estimations (using Hutchinson's trace estimator).

---

In [23]:
# Instantaneous change of variables, checked against a finite-difference flow map.
f_ode = lambda z: np.array([-0.5 * z[0] + 0.3 * np.sin(z[1]), 0.4 * z[0] - 0.2 * z[1]])
div = lambda z: -0.5 - 0.2  # trace of the Jacobian, constant here
z0, dt = np.array([0.7, -0.3]), 1e-4
step = lambda z: z + dt * f_ode(z)
Jnum = np.column_stack([(step(z0 + 1e-6 * e) - step(z0 - 1e-6 * e)) / 2e-6 for e in np.eye(2)])
dlogp = -np.log(abs(np.linalg.det(Jnum))) / dt
print(f"-(1/dt) log|det d(step)| = {dlogp:.8f}   -Tr J_f = {-div(z0):.8f}")
assert abs(dlogp + div(z0)) < 1e-3

-(1/dt) log|det d(step)| = 0.70002600   -Tr J_f = 0.70000000


### Problem L2.7 — Marginal Distribution of Bivariate Gaussian via Double Integration

**Source**: Pattern Recognition & Machine Learning / Probability Theory

**Statement**
Let $(X,Y)$ follow a bivariate normal distribution centered at origin with variance 1 and correlation $\rho \in (-1,1)$:

$$
p(x,y) = \frac{1}{2\pi \sqrt{1-\rho^2}} \exp\left( -\frac{x^2 - 2\rho x y + y^2}{2(1-\rho^2)} \right)
$$

Derive the marginal probability density $p_X(x) = \int_{-\infty}^\infty p(x,y) \, dy$.

**Intuition**

Marginalizing out a continuous variable is an integration operation over its coordinate axis. Completing the square isolates $y$ into a standard Gaussian integral.

**Solution**

1. Rewrite quadratic exponent in terms of $y$:

$$
x^2 - 2\rho x y + y^2 = (y - \rho x)^2 + (1-\rho^2) x^2
$$

2. Substitute into $p(x,y)$:

$$
p(x,y) = \frac{1}{2\pi \sqrt{1-\rho^2}} \exp\left( -\frac{x^2}{2} \right) \exp\left( -\frac{(y - \rho x)^2}{2(1-\rho^2)} \right)
$$

3. Integrate over $y \in (-\infty, \infty)$:

$$
p_X(x) = \frac{e^{-\frac{x^2}{2}}}{2\pi \sqrt{1-\rho^2}} \int_{-\infty}^\infty \exp\left( -\frac{(y - \rho x)^2}{2(1-\rho^2)} \right) dy
$$

4. The integral over $y$ is a 1D Gaussian integral with standard deviation $\sigma_y = \sqrt{1-\rho^2}$, which integrates to $\sqrt{2\pi(1-\rho^2)}$.
5. Multiply:

$$
p_X(x) = \frac{e^{-\frac{x^2}{2}}}{2\pi \sqrt{1-\rho^2}} \cdot \sqrt{2\pi(1-\rho^2)} = \frac{1}{\sqrt{2\pi}} e^{-\frac{x^2}{2}}
$$

$$
\boxed{p_X(x) = \frac{1}{\sqrt{2\pi}} e^{-\frac{x^2}{2}} \sim \mathcal{N}(0,1)}
$$

**Key takeaway**

Marginal distributions of multivariate Gaussians are strictly Gaussian, obtained by completing the square under multivariable integrals.

---

In [24]:
# Marginal of a bivariate normal with correlation rho = 0.6.
rho_c = 0.6
p = lambda x, y: np.exp(-(x**2 - 2 * rho_c * x * y + y**2) / (2 * (1 - rho_c**2))) \
    / (2 * np.pi * np.sqrt(1 - rho_c**2))
for x0 in (-1.0, 0.0, 1.3):
    marg = integrate.quad(lambda y: p(x0, y), -40, 40)[0]
    std = np.exp(-x0**2 / 2) / np.sqrt(2 * np.pi)
    print(f"x = {x0:5.2f}   marginal = {marg:.12f}   N(0,1) pdf = {std:.12f}")
    assert abs(marg - std) < 1e-10

x = -1.00   marginal = 0.241970724519   N(0,1) pdf = 0.241970724519
x =  0.00   marginal = 0.398942280401   N(0,1) pdf = 0.398942280401
x =  1.30   marginal = 0.171368592048   N(0,1) pdf = 0.171368592048


### Problem L2.8 — Electrostatic Field on Axis of Uniformly Charged Disk

**Source**: Jackson, *Classical Electrodynamics*, Ch. 1

**Statement**
A flat thin disk of radius $R$ carries a uniform surface charge density $\sigma_0$. Compute the electrostatic potential $V(z)$ at a point $(0,0,z)$ on its central axis.

**Intuition**

Integrate the potential contributions $dV = \frac{1}{4\pi \varepsilon_0} \frac{dq}{d}$ from concentric charge rings of radius $r$ and width $dr$.

**Solution**

1. Differential charge element $dq = \sigma_0 dA = \sigma_0 r \, dr \, d\theta$.
2. Distance from disk element $(r, \theta, 0)$ to axis point $(0, 0, z)$ is $d = \sqrt{r^2 + z^2}$.
3. Potential integral:

$$
\begin{aligned}
V(z) &= \frac{1}{4\pi \varepsilon_0} \int_0^{2\pi} \int_0^R \frac{\sigma_0 r \, dr \, d\theta}{\sqrt{r^2 + z^2}} \\
&= \frac{2\pi \sigma_0}{4\pi \varepsilon_0} \int_0^R \frac{r}{\sqrt{r^2 + z^2}} \, dr \\
&= \frac{\sigma_0}{2\varepsilon_0} \Big[ \sqrt{r^2 + z^2} \Big]_0^R = \frac{\sigma_0}{2\varepsilon_0} \left( \sqrt{R^2 + z^2} - \lvert z \rvert \right)
\end{aligned}
$$

$$
\boxed{V(z) = \frac{\sigma_0}{2\varepsilon_0} \left( \sqrt{R^2 + z^2} - \lvert z \rvert \right)}
$$

**Key takeaway**

As $z \to \infty$, Taylor expansion gives $V(z) \approx \frac{\sigma_0 \pi R^2}{4\pi \varepsilon_0 z} = \frac{Q}{4\pi \varepsilon_0 z}$, recovering point charge potential.

---

In [25]:
# Potential on the axis of a uniformly charged disk, sigma0 = eps0 = 1.
R = 2.0
for z0 in (0.0, 0.5, 3.0):
    num, _ = integrate.nquad(lambda r, th: r / np.sqrt(r**2 + z0**2) / (4 * np.pi),
                             [[0, R], [0, 2 * np.pi]])
    hand = 0.5 * (np.sqrt(R**2 + z0**2) - abs(z0))
    print(f"z = {z0:4.1f}   numeric V = {num:.12f}   closed form = {hand:.12f}")
    assert abs(num - hand) < 1e-9

z =  0.0   numeric V = 1.000000000000   closed form = 1.000000000000
z =  0.5   numeric V = 0.780776406404   closed form = 0.780776406404
z =  3.0   numeric V = 0.302775637732   closed form = 0.302775637732


### Problem L2.9 — Differential Entropy of Multivariate Gaussian

**Source**: Cover & Thomas, *Elements of Information Theory*, Ch. 8

**Statement**
Compute the differential entropy $H(\mathbf{X}) = -\int_{\mathbb{R}^n} p(\mathbf{x}) \log p(\mathbf{x}) \, d^n\mathbf{x}$ of an $n$-dimensional multivariate Gaussian distribution $\mathbf{X} \sim \mathcal{N}(\boldsymbol{\mu}, \mathbf{\Sigma})$.

**Intuition**

Entropy measures average surprise. Substituting $\log p(\mathbf{x})$ converts the integral into an expectation of a quadratic form over a Gaussian measure.

**Solution**

1. Probability density: $p(\mathbf{x}) = (2\pi)^{-\frac{n}{2}} (\det \mathbf{\Sigma})^{-\frac{1}{2}} \exp\left( -\frac{1}{2} (\mathbf{x}-\boldsymbol{\mu})^T \mathbf{\Sigma}^{-1} (\mathbf{x}-\boldsymbol{\mu}) \right)$.
2. Take log:

$$
\log p(\mathbf{x}) = -\frac{n}{2} \log(2\pi) - \frac{1}{2} \log(\det \mathbf{\Sigma}) - \frac{1}{2} (\mathbf{x}-\boldsymbol{\mu})^T \mathbf{\Sigma}^{-1} (\mathbf{x}-\boldsymbol{\mu})
$$

3. Take negative expectation $H(\mathbf{X}) = -\mathbb{E}[\log p(\mathbf{X})]$:

$$
H(\mathbf{X}) = \frac{n}{2} \log(2\pi) + \frac{1}{2} \log(\det \mathbf{\Sigma}) + \frac{1}{2} \mathbb{E}\left[ (\mathbf{X}-\boldsymbol{\mu})^T \mathbf{\Sigma}^{-1} (\mathbf{X}-\boldsymbol{\mu}) \right]
$$

4. Using the trace trick for expectation of quadratic form:

$$
\mathbb{E}\left[ (\mathbf{X}-\boldsymbol{\mu})^T \mathbf{\Sigma}^{-1} (\mathbf{X}-\boldsymbol{\mu}) \right] = \operatorname{Tr}\left( \mathbf{\Sigma}^{-1} \mathbb{E}[(\mathbf{X}-\boldsymbol{\mu})(\mathbf{X}-\boldsymbol{\mu})^T] \right) = \operatorname{Tr}(\mathbf{\Sigma}^{-1} \mathbf{\Sigma}) = \operatorname{Tr}(\mathbf{I}_n) = n
$$

5. Combine terms:

$$
H(\mathbf{X}) = \frac{1}{2} \log\left( (2\pi)^n \det \mathbf{\Sigma} \right) + \frac{n}{2} = \frac{1}{2} \log\left( (2\pi e)^n \det \mathbf{\Sigma} \right)
$$

$$
\boxed{H(\mathbf{X}) = \frac{1}{2} \log\left( (2\pi e)^n \det \mathbf{\Sigma} \right)}
$$

**Key takeaway**

The determinant of the covariance matrix $\det \mathbf{\Sigma}$ represents the total volume of uncertainty in $n$-dimensional state space.

---

In [26]:
# Differential entropy of a 2-D Gaussian: quadrature versus (1/2) log((2 pi e)^n det Sigma).
Sigma = np.array([[2.0, 0.7], [0.7, 1.0]])
P = np.linalg.inv(Sigma)
logZ = 0.5 * np.log((2 * np.pi) ** 2 * np.linalg.det(Sigma))
q = lambda x, y: P[0, 0] * x * x + 2 * P[0, 1] * x * y + P[1, 1] * y * y
pdf = lambda x, y: np.exp(-0.5 * q(x, y) - logZ)
# -p log p = p (q/2 + log Z), written out so no logarithm of an underflowed density is taken.
H, _ = integrate.dblquad(lambda y, x: pdf(x, y) * (0.5 * q(x, y) + logZ), -15, 15, -15, 15)
hand = 0.5 * np.log((2 * np.pi * np.e) ** 2 * np.linalg.det(Sigma))
print(f"quadrature H = {H:.10f}   (1/2) log((2 pi e)^n det Sigma) = {hand:.10f}")
assert abs(H - hand) < 1e-8

quadrature H = 3.0439318918   (1/2) log((2 pi e)^n det Sigma) = 3.0439318918


### Problem L2.10 — Volume of Arbitrary Ellipsoid via Linear Jacobian

**Source**: Marsden & Tromba, *Vector Calculus*, Ch. 6.2

**Statement**
Calculate the volume of the general 3D ellipsoid $\frac{x^2}{a^2} + \frac{y^2}{b^2} + \frac{z^2}{c^2} \le 1$ using the linear transformation change of variables formula.

**Intuition**

Map the ellipsoid to the unit sphere $u^2+v^2+w^2 \le 1$. The Jacobian determinant scales the known sphere volume $\frac{4}{3}\pi$.

**Solution**

1. **Define Map**: $x = au, y = bv, z = cw$.
2. **Jacobian**:

$$
J = \begin{pmatrix} a & 0 & 0 \\ 0 & b & 0 \\ 0 & 0 & c \end{pmatrix} \implies \det J = abc
$$

3. **Apply Change of Variables**:

$$
V = \iiint_{\text{Ellipsoid}} dx \, dy \, dz = \iiint_{u^2+v^2+w^2 \le 1} abc \, du \, dv \, dw = abc \, V(B_3(1))
$$

4. Since volume of unit sphere $V(B_3(1)) = \frac{4}{3}\pi$:

$$
V = \frac{4}{3}\pi a b c
$$

$$
\boxed{V = \frac{4}{3}\pi a b c}
$$

**Key takeaway**

Linear transformation Jacobians act as global volumetric scaling factors across geometric domains.

---

In [27]:
# Ellipsoid volume by quadrature, a=3, b=2, c=1.5.
a, b, c = 3.0, 2.0, 1.5
vol, _ = integrate.tplquad(lambda w, v, u: a * b * c, -1, 1,
                           lambda u: -np.sqrt(1 - u**2), lambda u: np.sqrt(1 - u**2),
                           lambda u, v: -np.sqrt(max(1 - u**2 - v**2, 0.0)),
                           lambda u, v: np.sqrt(max(1 - u**2 - v**2, 0.0)))
print(f"tplquad = {vol:.10f}   4 pi a b c / 3 = {4 / 3 * np.pi * a * b * c:.10f}")
assert abs(vol - 4 / 3 * np.pi * a * b * c) < 1e-6

tplquad = 37.6991118431   4 pi a b c / 3 = 37.6991118431


### Problem L2.11 — Expectation of Quadratic Form via Multidimensional Integrals

**Source**: Advanced Statistical Learning & Matrix Calculus

**Statement**
Let $\mathbf{x} \sim \mathcal{N}(\mathbf{0}, \mathbf{\Sigma})$ in $\mathbb{R}^n$. Compute $\mathbb{E}[\mathbf{x}^T \mathbf{A} \mathbf{x}]$ for any symmetric matrix $\mathbf{A}$.

**Intuition**

Express the scalar quadratic form $\mathbf{x}^T \mathbf{A} \mathbf{x}$ as a sum $\sum_{i,j} A_{ij} x_i x_j$ and use linearity of expectation over Gaussian integrals.

**Solution**

1. **Summation Expansion**:

$$
\mathbb{E}[\mathbf{x}^T \mathbf{A} \mathbf{x}] = \mathbb{E}\left[ \sum_{i=1}^n \sum_{j=1}^n A_{ij} x_i x_j \right] = \sum_{i=1}^n \sum_{j=1}^n A_{ij} \mathbb{E}[x_i x_j]
$$

2. By definition of covariance for zero-mean vector: $\mathbb{E}[x_i x_j] = \Sigma_{ij}$.
3. Substitute $\Sigma_{ij}$:

$$
\mathbb{E}[\mathbf{x}^T \mathbf{A} \mathbf{x}] = \sum_{i=1}^n \sum_{j=1}^n A_{ij} \Sigma_{ji} = \operatorname{Tr}(\mathbf{A} \mathbf{\Sigma})
$$

$$
\boxed{\mathbb{E}[\mathbf{x}^T \mathbf{A} \mathbf{x}] = \operatorname{Tr}(\mathbf{A} \mathbf{\Sigma})}
$$

**Key takeaway**

Integrals of quadratic forms against Gaussian densities simplify directly to matrix trace inner products $\operatorname{Tr}(\mathbf{A}\mathbf{\Sigma})$.

---

In [28]:
# E[x^T A x] = Tr(A Sigma) by sampling.
Sigma = np.array([[2.0, 0.6, 0.1], [0.6, 1.4, -0.3], [0.1, -0.3, 0.9]])
A = np.array([[1.0, 0.2, -0.5], [0.2, 3.0, 0.4], [-0.5, 0.4, 0.7]])
L = np.linalg.cholesky(Sigma)
X = rng.normal(size=(4_000_000, 3)) @ L.T
emp = np.einsum("ni,ij,nj->n", X, A, X).mean()
print(f"empirical E[x^T A x] = {emp:.6f}   Tr(A Sigma) = {np.trace(A @ Sigma):.6f}")
assert abs(emp - np.trace(A @ Sigma)) < 0.02

empirical E[x^T A x] = 6.729008   Tr(A Sigma) = 6.730000


## L3 — Challenge Proofs

### Problem L3.1 — A non-linear substitution that flattens a triangle

**Source**: Classical change-of-variables exercise; Apostol, *Calculus, Vol. II*, 2nd ed., §11.28

**Statement**
Evaluate the double integral:

$$
I = \int_0^1 \int_0^{1-x} e^{\frac{y}{x+y}} \, dy \, dx
$$

**Intuition**

The argument of the exponent $\frac{y}{x+y}$ suggests a coordinate transformation that isolates $x+y$ and $\frac{y}{x+y}$ into independent variables.

**Solution**

1. **Define Transformation**:
   Let $u = x + y$ and $v = \frac{y}{x+y}$.
2. **Find Inverse Map**:
   $y = u v$, and $x = u - y = u(1-v)$.
3. **Determine Domain in $(u,v)$-space**:
   - Original region $D$: $x \ge 0, y \ge 0, x + y \le 1$.
   - $u = x+y$ ranges from $0$ to $1$.
   - $v = \frac{y}{x+y}$ ranges from $0$ (when $y=0$) to $1$ (when $x=0$).
   - The transformation maps $D$ to a rectangular domain $R = [0,1] \times [0,1]$ in $(u,v)$-space!
4. **Compute Jacobian**:

$$
J = \begin{pmatrix} \frac{\partial x}{\partial u} & \frac{\partial x}{\partial v} \\ \frac{\partial y}{\partial u} & \frac{\partial y}{\partial v} \end{pmatrix} = \begin{pmatrix} 1-v & -u \\ v & u \end{pmatrix}
$$

$$
\det J = (1-v)u - (-u)v = u - uv + uv = u
$$

Since $u \in [0,1]$, $\lvert\det J\rvert = u$.
5. **Transform and Evaluate Integral**:

$$
\begin{aligned}
I &= \int_0^1 \int_0^1 e^v \cdot u \, du \, dv \\
&= \left( \int_0^1 e^v dv \right) \left( \int_0^1 u \, du \right) \\
&= (e - 1) \cdot \left[ \frac{u^2}{2} \right]_0^1 = \frac{e - 1}{2}
\end{aligned}
$$

$$
\boxed{I = \frac{e - 1}{2}}
$$

**Key takeaway**

Non-linear coordinate mappings can flatten triangular domains into hyper-rectangles while making rational exponents linear.

---

In [29]:
# exp(y/(x+y)) over the triangle x, y >= 0, x + y <= 1.
direct = integrate.quad(
    lambda x: integrate.quad(lambda y: np.exp(y / (x + y)) if x + y > 0 else 1.0, 0, 1 - x)[0],
    0, 1)[0]
print(f"direct = {direct:.10f}   (e-1)/2 = {(np.e - 1) / 2:.10f}")
assert abs(direct - (np.e - 1) / 2) < 1e-6

direct = 0.8591409142   (e-1)/2 = 0.8591409142


### Problem L3.2 — The Basel value $\zeta(2)$ from the unit square

**Source**: Apostol, *A Proof that Euler Missed*, Math. Intelligencer 5 (1983), pp. 59–60

**Statement**
Find the value of $\int_0^1 \int_0^1 \frac{1}{1 - xy} \, dx \, dy$ as an infinite series, and express it in terms of $\zeta(2) = \frac{\pi^2}{6}$.

**Intuition**

Expand the integrand $\frac{1}{1-xy}$ as a geometric series $\sum_{k=0}^\infty (xy)^k$ and integrate term-by-term using Fubini's theorem.

**Solution**

1. **Geometric Series Expansion**:
   For $x, y \in (0,1)$, $0 \lt xy \lt 1$.

$$
\frac{1}{1 - xy} = \sum_{k=0}^\infty (xy)^k
$$

2. **Apply Fubini's Theorem** (all terms are positive):

$$
\int_0^1 \int_0^1 \frac{1}{1 - xy} \, dx \, dy = \sum_{k=0}^\infty \int_0^1 \int_0^1 x^k y^k \, dx \, dy
$$

3. **Evaluate Term-by-Term**:

$$
\int_0^1 x^k dx = \frac{1}{k+1}, \quad \int_0^1 y^k dy = \frac{1}{k+1}
$$

$$
\int_0^1 \int_0^1 x^k y^k \, dx \, dy = \frac{1}{(k+1)^2}
$$

4. **Sum the Series**:

$$
I = \sum_{k=0}^\infty \frac{1}{(k+1)^2} = \sum_{n=1}^\infty \frac{1}{n^2} = \zeta(2) = \frac{\pi^2}{6}
$$

$$
\boxed{I = \frac{\pi^2}{6}}
$$

**Key takeaway**

Double integrals over the unit square can generate fundamental special values of the Riemann zeta function.

---

In [30]:
# The Basel double integral.
val = integrate.quad(lambda y: integrate.quad(lambda x: 1 / (1 - x * y), 0, 1)[0], 0, 1)[0]
partial = sum(1 / k**2 for k in range(1, 200_001))
print(f"quadrature = {val:.10f}   pi^2/6 = {np.pi**2 / 6:.10f}   partial sum = {partial:.10f}")
assert abs(val - np.pi**2 / 6) < 1e-6

quadrature = 1.6449340668   pi^2/6 = 1.6449340668   partial sum = 1.6449290669


### Problem L3.3 — Dirichlet's Integral & $n$-Simplex Volume

**Source**: Demidovich No. 4101 / Dirichlet (1839)

**Statement**
Prove Dirichlet's integral formula for the volume of the standard $n$-dimensional simplex $T_n = \{\mathbf{x} \in \mathbb{R}^n : x_i \ge 0, \, \sum_{i=1}^n x_i \le 1\}$:

$$
V(T_n) = \int \dots \int_{T_n} dx_1 dx_2 \dots dx_n = \frac{1}{n!}
$$

**Intuition**

Prove by mathematical induction on dimension $n$, integrating out one coordinate axis at each step.

**Solution**

1. **Base Case $n=1$**:
   $T_1 = \{x_1 \ge 0, x_1 \le 1\} \implies V(T_1) = \int_0^1 dx_1 = 1 = \frac{1}{1!}$. True!
2. **Inductive Hypothesis**:
   Assume $V(T_{k-1}) = \frac{(1 - x_k)^{k-1}}{(k-1)!}$ when the simplex sum is bounded by $1 - x_k$.
3. **Inductive Step**:
   For $T_k$, $x_k$ ranges from $0$ to $1$. For a fixed $x_k$, the remaining $k-1$ variables satisfy $\sum_{i=1}^{k-1} x_i \le 1 - x_k$.
   By linear scaling, the $(k-1)$-dimensional volume of this cross-section is $V(T_{k-1}) \cdot (1 - x_k)^{k-1} = \frac{(1 - x_k)^{k-1}}{(k-1)!}$.
4. Integrate along $x_k$:

$$
V(T_k) = \int_0^1 \frac{(1 - x_k)^{k-1}}{(k-1)!} \, dx_k = \frac{1}{(k-1)!} \left[ -\frac{(1 - x_k)^k}{k} \right]_0^1 = \frac{1}{k!}
$$

By induction, $V(T_n) = \frac{1}{n!}$ for all $n \ge 1$.

$$
\boxed{V(T_n) = \frac{1}{n!}}
$$

**Key takeaway**

The volume of an $n$-dimensional simplex bounded by coordinate planes and a hyper-plane is exactly $\frac{1}{n!}$.

---

In [31]:
# Simplex volume 1/n! by Monte Carlo in n = 2..6.
for n in range(2, 7):
    U = rng.random((2_000_000, n))
    frac = (U.sum(axis=1) <= 1.0).mean()
    print(f"n = {n}   MC volume = {frac:.8f}   1/n! = {1 / math.factorial(n):.8f}")
    assert abs(frac - 1 / math.factorial(n)) < 5e-3

n = 2   MC volume = 0.50036200   1/n! = 0.50000000


n = 3   MC volume = 0.16670300   1/n! = 0.16666667


n = 4   MC volume = 0.04169300   1/n! = 0.04166667
n = 5   MC volume = 0.00827950   1/n! = 0.00833333


n = 6   MC volume = 0.00138800   1/n! = 0.00138889


### Problem L3.4 — Derivation of $n$-Sphere Volume and Surface Area

**Source**: Apostol, *Calculus, Vol. II*, 2nd ed., §11.28

**Statement**
Using the Gaussian integral identity $I_n = \int_{\mathbb{R}^n} e^{-\|\mathbf{x}\|^2} d^n\mathbf{x} = \pi^{\frac{n}{2}}$, derive the exact formulas for the surface area $S_{n-1}$ and volume $V_n$ of the unit $n$-ball.

**Intuition**

Express the $n$-dimensional Gaussian integral in spherical hypershells $r = \|\mathbf{x}\|$, then solve for $S_{n-1}$ via Gamma function substitution.

**Solution**

1. **Cartesian Gaussian Value**:

$$
I_n = \prod_{i=1}^n \int_{-\infty}^\infty e^{-x_i^2} dx_i = (\sqrt{\pi})^n = \pi^{\frac{n}{2}}
$$

2. **Hyperspherical Slicing**:
   Slice $\mathbb{R}^n$ into hyperspherical shells of radius $r$ with surface area $S_{n-1} r^{n-1}$:

$$
I_n = \int_0^\infty e^{-r^2} S_{n-1} r^{n-1} \, dr = S_{n-1} \int_0^\infty r^{n-1} e^{-r^2} \, dr
$$

3. **Gamma Substitution**: Let $u = r^2 \implies r = u^{\frac{1}{2}}, dr = \frac{1}{2}u^{-\frac{1}{2}}du$.

$$
\int_0^\infty r^{n-1} e^{-r^2} \, dr = \frac{1}{2} \int_0^\infty u^{\frac{n}{2}-1} e^{-u} \, du = \frac{1}{2} \Gamma\left(\frac{n}{2}\right)
$$

4. Equate:

$$
\pi^{\frac{n}{2}} = S_{n-1} \cdot \frac{1}{2} \Gamma\left(\frac{n}{2}\right) \implies S_{n-1} = \frac{2\pi^{\frac{n}{2}}}{\Gamma\left(\frac{n}{2}\right)}
$$

5. Integrate surface area to get unit ball volume $V_n$:

$$
V_n = \int_0^1 S_{n-1} r^{n-1} \, dr = \frac{S_{n-1}}{n} = \frac{2\pi^{\frac{n}{2}}}{n \Gamma\left(\frac{n}{2}\right)} = \frac{\pi^{\frac{n}{2}}}{\Gamma\left(\frac{n}{2} + 1\right)}
$$

$$
\boxed{V_n(R) = \frac{\pi^{\frac{n}{2}}}{\Gamma\left(\frac{n}{2} + 1\right)} R^n, \quad S_{n-1}(R) = \frac{2\pi^{\frac{n}{2}}}{\Gamma\left(\frac{n}{2}\right)} R^{n-1}}
$$

**Key takeaway**

Connecting multivariable integration in Cartesian coordinates to hyperspherical coordinates yields closed-form formulas for higher-dimensional geometry.

---

In [32]:
# n-ball volume and sphere surface area against nquad in low dimensions.
Vn = lambda n: np.pi ** (n / 2) / special.gamma(n / 2 + 1)
Sn = lambda n: 2 * np.pi ** (n / 2) / special.gamma(n / 2)
for n, ref in ((1, 2.0), (2, np.pi), (3, 4 * np.pi / 3)):
    print(f"n = {n}   V_n = {Vn(n):.12f}   known = {ref:.12f}   S_(n-1) = {Sn(n):.12f}")
    assert abs(Vn(n) - ref) < 1e-12
    assert abs(Sn(n) - n * Vn(n)) < 1e-12
gauss = integrate.quad(lambda r: Sn(5) * r**4 * np.exp(-r * r), 0, np.inf)[0]
print(f"shell integral in n = 5 = {gauss:.12f}   pi^(5/2) = {np.pi**2.5:.12f}")
assert abs(gauss - np.pi**2.5) < 1e-10

n = 1   V_n = 2.000000000000   known = 2.000000000000   S_(n-1) = 2.000000000000
n = 2   V_n = 3.141592653590   known = 3.141592653590   S_(n-1) = 6.283185307180
n = 3   V_n = 4.188790204786   known = 4.188790204786   S_(n-1) = 12.566370614359
shell integral in n = 5 = 17.493418327625   pi^(5/2) = 17.493418327625


### Problem L3.5 — Order swap that removes an exponential-integral

**Source**: Demidovich, *Problems in Mathematical Analysis*, order-reversal section (improper double integrals)

**Statement**
Evaluate the improper double integral:

$$
I = \int_0^\infty \int_x^\infty \frac{e^{-y}}{y} \, dy \, dx
$$

**Intuition**

Evaluating the inner integral $\int_x^\infty \frac{e^{-y}}{y} dy$ involves the non-elementary exponential integral function $E_1(x)$. Reversing the order of integration clears the denominator!

**Solution**

1. **Identify domain $D$**:
   $0 \le x \lt \infty, \quad x \le y \lt \infty$.
   This is the region in the first quadrant above the line $y = x$.
2. **Reverse integration order**:
   Outer variable $y \in [0, \infty)$. For fixed $y$, $x$ ranges from $0$ to $y$.
3. **Rewrite integral**:

$$
I = \int_0^\infty \int_0^y \frac{e^{-y}}{y} \, dx \, dy
$$

4. **Evaluate inner integral**:

$$
\int_0^y \frac{e^{-y}}{y} \, dx = \frac{e^{-y}}{y} \Big[ x \Big]_0^y = \frac{e^{-y}}{y} \cdot y = e^{-y}
$$

5. **Evaluate outer integral**:

$$
I = \int_0^\infty e^{-y} \, dy = \left[ -e^{-y} \right]_0^\infty = 0 - (-1) = 1
$$

$$
\boxed{I = 1}
$$

**Key takeaway**

Order swapping can simplify expressions containing special functions into elementary integrations.

---

In [33]:
# Order swap for exp(-y)/y over the wedge 0 <= x <= y.
swapped = integrate.quad(lambda y: np.exp(-y), 0, np.inf)[0]
direct = integrate.quad(
    lambda x: integrate.quad(lambda y: np.exp(-y) / y, x, np.inf)[0], 1e-9, 60)[0]
print(f"swapped order = {swapped:.10f}   direct order (x from 1e-9) = {direct:.6f}")
assert abs(swapped - 1.0) < 1e-10 and abs(direct - 1.0) < 1e-3

swapped order = 1.0000000000   direct order (x from 1e-9) = 1.000000


### Problem L3.6 — Counterexample to Fubini's Theorem

**Source**: Apostol, *Mathematical Analysis*, Ch. 14 / Kaczor & Nowak

**Statement**
Let $f(x,y) = \frac{x^2 - y^2}{(x^2 + y^2)^2}$ on $(0,1] \times (0,1]$.
1. Compute $\int_0^1 \left( \int_0^1 f(x,y) \, dy \right) dx$.
2. Compute $\int_0^1 \left( \int_0^1 f(x,y) \, dx \right) dy$.
3. Explain why Fubini's theorem does not apply.

**Intuition**

Fubini's theorem requires $\iint \lvert f(x,y) \rvert dA \lt \infty$. If absolute integrability fails, iterated integrals can yield completely different values.

**Solution**

1. **Inner integral with respect to $y$**:
   Note that $\frac{\partial}{\partial y} \left( \frac{y}{x^2+y^2} \right) = \frac{(x^2+y^2) - y(2y)}{(x^2+y^2)^2} = \frac{x^2 - y^2}{(x^2+y^2)^2}$.

$$
\int_0^1 \frac{x^2 - y^2}{(x^2+y^2)^2} \, dy = \left[ \frac{y}{x^2+y^2} \right]_{y=0}^{y=1} = \frac{1}{x^2+1}
$$

2. **Outer integral with respect to $x$**:

$$
\int_0^1 \frac{dx}{x^2+1} = \Big[ \arctan x \Big]_0^1 = \frac{\pi}{4}
$$

3. **Inner integral with respect to $x$**:
   By skew-symmetry $f(y,x) = -f(x,y)$:

$$
\int_0^1 \frac{x^2 - y^2}{(x^2+y^2)^2} \, dx = \left[ -\frac{x}{x^2+y^2} \right]_{x=0}^{x=1} = -\frac{1}{y^2+1}
$$

4. **Outer integral with respect to $y$**:

$$
\int_0^1 \left(-\frac{1}{y^2+1}\right) dy = -\frac{\pi}{4}
$$

5. Since $\frac{\pi}{4} \ne -\frac{\pi}{4}$, Fubini's theorem fails because $\iint_{[0,1]^2} \lvert f(x,y) \rvert \, dA = +\infty$.

$$
\boxed{\int_0^1 \int_0^1 f \, dy \, dx = \frac{\pi}{4}, \quad \int_0^1 \int_0^1 f \, dx \, dy = -\frac{\pi}{4}. \text{ Fubini fails because } \iint \lvert f \rvert dA = \infty.}
$$

**Key takeaway**

Absolute integrability is an indispensable hypothesis for Fubini's theorem; conditionally convergent multivariable integrals depend on integration order.

---

In [34]:
# The two iterated integrals of (x^2 - y^2)/(x^2 + y^2)^2 on (0,1]^2.
g = lambda x, y: (x * x - y * y) / (x * x + y * y) ** 2
dy_dx = integrate.quad(lambda x: integrate.quad(lambda y: g(x, y), 0, 1, limit=400)[0],
                       0, 1, limit=400)[0]
dx_dy = integrate.quad(lambda y: integrate.quad(lambda x: g(x, y), 0, 1, limit=400)[0],
                       0, 1, limit=400)[0]
mass = integrate.quad(lambda x: integrate.quad(lambda y: abs(g(x, y)), x, 1, limit=400)[0],
                      1e-6, 1, limit=400)[0]
print(f"dy dx = {dy_dx:.10f}   pi/4 = {np.pi / 4:.10f}")
print(f"dx dy = {dx_dy:.10f}  -pi/4 = {-np.pi / 4:.10f}")
print(f"absolute mass above the diagonal, cut off at x = 1e-6: {mass:.4f} (grows like log(1/eps))")
assert abs(dy_dx - np.pi / 4) < 1e-6 and abs(dx_dy + np.pi / 4) < 1e-6 and mass > 5

dy dx = 0.7853981634   pi/4 = 0.7853981634
dx dy = -0.7853981634  -pi/4 = -0.7853981634
absolute mass above the diagonal, cut off at x = 1e-6: 6.1224 (grows like log(1/eps))


/tmp/ipykernel_270344/324620223.py:7: IntegrationWarning: The algorithm does not converge.  Roundoff error is detected
  in the extrapolation table.  It is assumed that the requested tolerance
  cannot be achieved, and that the returned result (if full_output = 1) is 
  the best which can be obtained.
  mass = integrate.quad(lambda x: integrate.quad(lambda y: abs(g(x, y)), x, 1, limit=400)[0],


### Problem L3.7 — Rotating a linear form onto an axis in the unit ball

**Source**: Marsden & Tromba, *Vector Calculus*, 6th ed., §6.3 (rotational symmetry of the ball)

**Statement**
Find the value of the triple integral over the unit ball $B_3 = \{(x,y,z) : x^2+y^2+z^2 \le 1\}$:

$$
I = \iiint_{B_3} (1 + x + y + z)^{10} (x^2+y^2+z^2) \, dx \, dy \, dz
$$

**Intuition**

Examine the symmetries of the domain $B_3$. The region $B_3$ is invariant under any orthogonal rotation of space.

**Solution**

1. **Symmetry Transformation**:
   Rotate coordinates so that the linear direction $(1,1,1)^T$ aligns with the $z'$-axis.
   Let $\mathbf{u} = \frac{1}{\sqrt{3}}(1,1,1)^T$. Then $x+y+z = \sqrt{3} (\mathbf{u} \cdot \mathbf{x}) = \sqrt{3} z'$.
   Under orthogonal rotation, $x^2+y^2+z^2 = {x'}^2 + {y'}^2 + {z'}^2 = \rho^2$.
2. **Transform to Spherical Coordinates** $(\rho, \phi, \theta)$ in the rotated frame:
   $z' = \rho \cos\phi$.
   $1 + x + y + z = 1 + \sqrt{3}\rho\cos\phi$.
3. **Integral Setup**:

$$
I = \int_0^{2\pi} d\theta \int_0^1 \int_0^\pi (1 + \sqrt{3}\rho\cos\phi)^{10} \rho^2 (\rho^2 \sin\phi) \, d\phi \, d\rho
$$

4. **Evaluate $\phi$-integral**:
   Let $u = 1 + \sqrt{3}\rho\cos\phi \implies du = -\sqrt{3}\rho\sin\phi \, d\phi$.
   When $\phi = 0 \implies u = 1 + \sqrt{3}\rho$. When $\phi = \pi \implies u = 1 - \sqrt{3}\rho$.

$$
\int_0^\pi (1 + \sqrt{3}\rho\cos\phi)^{10} \sin\phi \, d\phi = \frac{1}{\sqrt{3}\rho} \int_{1-\sqrt{3}\rho}^{1+\sqrt{3}\rho} u^{10} du = \frac{(1+\sqrt{3}\rho)^{11} - (1-\sqrt{3}\rho)^{11}}{11 \sqrt{3} \rho}
$$

5. **Multiply by $2\pi\rho^4\,d\rho$ and integrate $\rho \in [0,1]$**:

$$
I = \frac{2\pi}{11\sqrt3} \int_0^1 \rho^3 \left[ (1+\sqrt3\rho)^{11} - (1-\sqrt3\rho)^{11} \right] d\rho .
$$

6. **Expand binomially.** Only the odd powers survive the subtraction:

$$
(1+t)^{11} - (1-t)^{11} = 2\sum_{j=0}^{5} \binom{11}{2j+1} t^{2j+1},
\qquad t = \sqrt3\,\rho, \quad t^{2j+1} = 3^{j}\sqrt3\,\rho^{2j+1}.
$$

7. **Integrate term by term** using $\int_0^1 \rho^{2j+4}\,d\rho = \frac{1}{2j+5}$:

$$
I = \frac{4\pi}{11} \sum_{j=0}^{5} \binom{11}{2j+1} \frac{3^{j}}{2j+5}
= \frac{4\pi}{11}\left( \frac{11}{5} + \frac{495}{7} + 462 + 810 + \frac{4455}{13} + \frac{81}{5} \right)
= \frac{4\pi}{11}\cdot\frac{775232}{455}.
$$

$$
\boxed{I = \frac{3100928\,\pi}{5005} \approx 1946.4241}
$$

**Interpretation.** The naive guess $4\pi/13 \approx 0.97$ is wrong by three orders of magnitude:
the factor $(1+x+y+z)^{10}$ reaches $(1+\sqrt3)^{10} \approx 5.4\times10^{4}$ near the point
$\tfrac{1}{\sqrt3}(1,1,1)$, and that peak, not the average, dominates the integral. The code cell
below confirms the value symbolically and, independently of the rotation argument, by direct
Cartesian quadrature over the ball.

**Key takeaway**

Orthogonal coordinate rotations simplify arbitrary linear directions $(a,b,c)\cdot\mathbf{x}$ into single axis components.

---

In [35]:
# The rotated integral, exactly with sympy, and independently by Cartesian quadrature
# over the ball -- a route that never uses the rotation argument.
rho_s, phi_s = sp.symbols("rho varphi", positive=True)
expr = (1 + sp.sqrt(3) * rho_s * sp.cos(phi_s)) ** 10 * rho_s**2 * rho_s**2 * sp.sin(phi_s)
exact = sp.simplify(2 * sp.pi * sp.integrate(sp.integrate(expr, (phi_s, 0, sp.pi)), (rho_s, 0, 1)))
print("exact value:", exact, "=", float(exact))

sx = lambda x: np.sqrt(max(1 - x * x, 0.0))
sxy = lambda x, y: np.sqrt(max(1 - x * x - y * y, 0.0))
cart, err = integrate.tplquad(lambda z, y, x: (1 + x + y + z) ** 10 * (x * x + y * y + z * z),
                              -1, 1, lambda x: -sx(x), lambda x: sx(x),
                              lambda x, y: -sxy(x, y), lambda x, y: sxy(x, y))
print(f"Cartesian tplquad = {cart:.8f} (estimated error {err:.1e})")
print(f"the discarded guess 4 pi / 13 = {4 * np.pi / 13:.6f} is smaller by a factor {cart / (4 * np.pi / 13):.0f}")
assert abs(float(exact) - 3100928 * np.pi / 5005) < 1e-6
assert abs(cart - float(exact)) < 1e-4

exact value: 3100928*pi/5005 = 1946.4241007214566
Cartesian tplquad = 1946.42410078 (estimated error 1.8e-05)
the discarded guess 4 pi / 13 = 0.966644 is smaller by a factor 2014


### Problem L3.8 — Bivariate Gaussian Integral with Cross-Term

**Source**: Polya & Szego, *Problems and Theorems in Analysis*, Vol. I

**Statement**
Evaluate $\iint_{\mathbb{R}^2} e^{-(x^2 + 2b x y + y^2)} dx dy$ for $\lvert b \rvert \lt 1$.

**Intuition**

The quadratic form $x^2 + 2bxy + y^2 = \mathbf{x}^T \mathbf{A} \mathbf{x}$ has matrix:

$$
\mathbf{A} = \begin{pmatrix} 1 & b \\ b & 1 \end{pmatrix}
$$

Use Theorem 4.5 of `first_principles.ipynb` for multivariate Gaussian integrals.

**Solution**

1. Matrix:

$$
\mathbf{A} = \begin{pmatrix} 1 & b \\ b & 1 \end{pmatrix}
$$

2. Check positive definiteness:
   Determinant $\det \mathbf{A} = 1 - b^2 \gt 0$ (since $\lvert b \rvert \lt 1$). Trace $= 2 \gt 0$.
3. Apply formula $\int_{\mathbb{R}^n} e^{-\mathbf{x}^T \mathbf{A} \mathbf{x}} d^n\mathbf{x} = \frac{\pi^{\frac{n}{2}}}{\sqrt{\det \mathbf{A}}}$:
   Here $n=2$, so $\pi^{\frac{2}{2}} = \pi$.

$$
I = \frac{\pi}{\sqrt{\det \mathbf{A}}} = \frac{\pi}{\sqrt{1 - b^2}}
$$

$$
\boxed{\iint_{\mathbb{R}^2} e^{-(x^2 + 2b x y + y^2)} dx dy = \frac{\pi}{\sqrt{1 - b^2}}}
$$

**Key takeaway**

Coupled Gaussian integrals resolve directly to the square root of the matrix determinant of the quadratic form.

---

In [36]:
# Coupled Gaussian in the plane for several b.
for bb in (0.0, 0.5, -0.9):
    num, _ = integrate.dblquad(lambda y, x: np.exp(-(x * x + 2 * bb * x * y + y * y)),
                               -40, 40, -40, 40)
    hand = np.pi / np.sqrt(1 - bb**2)
    print(f"b = {bb:5.2f}   dblquad = {num:.10f}   pi/sqrt(1-b^2) = {hand:.10f}")
    assert abs(num - hand) < 1e-7

b =  0.00   dblquad = 3.1415926536   pi/sqrt(1-b^2) = 3.1415926536
b =  0.50   dblquad = 3.6275987286   pi/sqrt(1-b^2) = 3.6275987285


b = -0.90   dblquad = 7.2073078418   pi/sqrt(1-b^2) = 7.2073078415


### Problem L3.9 — Hypersphere Surface Area via Riemannian Metric Tensor

**Source**: Spivak, *Calculus on Manifolds* / Differential Geometry

**Statement**
Express the surface volume element $dS$ of the sphere $S^2$ in terms of the metric tensor determinant $\sqrt{\det g}$ derived from spherical coordinates $( \theta, \phi )$.

**Intuition**

In differential geometry, the surface area element of an embedded manifold is $dS = \sqrt{\det g} \, d\theta \, d\phi$, where $g_{ij} = \frac{\partial \mathbf{r}}{\partial u_i} \cdot \frac{\partial \mathbf{r}}{\partial u_j}$.

**Solution**

1. Parameterize sphere of radius $R$:
   $\mathbf{r}(\theta, \phi) = (R\sin\phi\cos\theta, \, R\sin\phi\sin\theta, \, R\cos\phi)$.
2. Tangent vectors:

$$
\mathbf{e}_\theta = \frac{\partial \mathbf{r}}{\partial \theta} = (-R\sin\phi\sin\theta, \, R\sin\phi\cos\theta, \, 0)
$$

$$
\mathbf{e}_\phi = \frac{\partial \mathbf{r}}{\partial \phi} = (R\cos\phi\cos\theta, \, R\cos\phi\sin\theta, \, -R\sin\phi)
$$

3. Metric tensor components $g_{ij} = \mathbf{e}_i \cdot \mathbf{e}_j$:
   $g_{\theta\theta} = \mathbf{e}_\theta \cdot \mathbf{e}_\theta = R^2 \sin^2\phi$.
   $g_{\phi\phi} = \mathbf{e}_\phi \cdot \mathbf{e}_\phi = R^2$.
   $g_{\theta\phi} = g_{\phi\theta} = \mathbf{e}_\theta \cdot \mathbf{e}_\phi = 0$.
4. Metric matrix:

$$
g = \begin{pmatrix} R^2 \sin^2\phi & 0 \\ 0 & R^2 \end{pmatrix} \implies \det g = R^4 \sin^2\phi
$$

5. Area element:

$$
dS = \sqrt{\det g} \, d\theta \, d\phi = R^2 \sin\phi \, d\theta \, d\phi
$$

$$
\boxed{dS = R^2 \sin\phi \, d\theta \, d\phi}
$$

**Key takeaway**

The metric tensor determinant $\sqrt{\det g}$ provides the general manifold formulation for Jacobian surface area elements.

---

In [37]:
# Metric-tensor area element on the sphere of radius R = 1.7.
R = 1.7
phi_s, th_s = sp.symbols("varphi theta", positive=True)
rvec = sp.Matrix([R * sp.sin(phi_s) * sp.cos(th_s),
                  R * sp.sin(phi_s) * sp.sin(th_s),
                  R * sp.cos(phi_s)])
E = sp.Matrix.hstack(rvec.diff(th_s), rvec.diff(phi_s))
g = sp.simplify(E.T * E)
print("metric g =", g.tolist(), "   sqrt(det g) =", sp.simplify(sp.sqrt(g.det())))
assert sp.simplify(g.det() - R**4 * sp.sin(phi_s) ** 2) == 0
area, _ = integrate.nquad(lambda phi, th: R**2 * np.sin(phi), [[0, np.pi], [0, 2 * np.pi]])
print(f"total area = {area:.10f}   4 pi R^2 = {4 * np.pi * R**2:.10f}")
assert abs(area - 4 * np.pi * R**2) < 1e-10

metric g = [[2.89*sin(varphi)**2, 0], [0, 2.89000000000000]]    sqrt(det g) = 2.89*Abs(sin(varphi))
total area = 36.3168110755   4 pi R^2 = 36.3168110755


### Problem L3.10 — Putnam 1989 A2 — $e^{\max(x^2,y^2)}$ over the unit square

**Source**: William Lowell Putnam Mathematical Competition 1989, Problem A2

**Statement**
Evaluate $\iint_R e^{\max(x^2, y^2)} dx dy$, where $R = [0,1] \times [0,1]$.

**Intuition**

Split the unit square $R$ along the diagonal $y = x$ into two symmetric triangles where $\max(x^2, y^2)$ takes a simple single-variable form.

**Solution**

1. **Split Region $R$**:
   - Region $R_1$: $0 \le y \le x \le 1 \implies \max(x^2, y^2) = x^2$.
   - Region $R_2$: $0 \le x \le y \le 1 \implies \max(x^2, y^2) = y^2$.
   By symmetry across $y=x$, $\iint_{R_1} e^{x^2} dA = \iint_{R_2} e^{y^2} dA$.
2. **Calculate Integral over $R_1$**:

$$
I_1 = \int_0^1 \int_0^x e^{x^2} \, dy \, dx = \int_0^1 x e^{x^2} \, dx
$$

3. Substitute $u = x^2 \implies du = 2x \, dx$:

$$
I_1 = \frac{1}{2} \int_0^1 e^u \, du = \frac{e - 1}{2}
$$

4. **Total Integral**:

$$
I = 2 I_1 = 2 \left( \frac{e - 1}{2} \right) = e - 1
$$

$$
\boxed{\iint_R e^{\max(x^2, y^2)} dx dy = e - 1}
$$

**Key takeaway**

Partitioning domains along piecewise boundary curves (such as $y=x$) decomposes non-smooth functions into simple, smooth sub-integrals.

In [38]:
# exp(max(x^2, y^2)) over the unit square, split along the diagonal y = x.
half, _ = integrate.dblquad(lambda y, x: np.exp(x * x), 0, 1, 0, lambda x: x)
split = 2 * half
raw, _ = integrate.dblquad(lambda y, x: np.exp(max(x * x, y * y)), 0, 1, 0, 1)
print(f"split at the diagonal = {split:.12f}   raw dblquad = {raw:.12f}   e - 1 = {np.e - 1:.12f}")
print("the raw call loses digits because the integrand has a kink on y = x")
assert abs(split - (np.e - 1)) < 1e-10 and abs(raw - (np.e - 1)) < 1e-7

split at the diagonal = 1.718281828459   raw dblquad = 1.718281814931   e - 1 = 1.718281828459
the raw call loses digits because the integrand has a kink on y = x
